> **SENSITIVITY CALIBRATION — FULL THETA + SCCM_CORRECTED, WITHOUT LAB010,
> WITH REPAIRED SAMPLING SIGNALS FOR LAB004/LAB007/LAB008**
>
> This SOURCE notebook reuses the complete full-theta calibration suite without modifying its
> preprocessing, model, objective, bounds, weights or multistart settings. Two interventions are
> isolated here: (1) LAB010 is excluded from every natural-matrix optimization and
> identifiability refit, and (2) the observed CO2 signal of LAB004, LAB007 and LAB008 is
> replaced by the canonical sampling-repaired versions produced by the microleaks /
> sampling-transients diagnostics. Every other observed row stays byte-identical to the frozen
> SCCM-corrected input, so the previous-signals versus repaired-signals comparison below
> isolates exactly that signal change. LAB012 remains the internal natural holdout, and LAB010
> is evaluated afterwards with frozen parameters. LAB013–LAB015 are external kinetic / activation holdouts and are never used to tune
> anything; their documented-unreliable gas signals are shown only in an exploratory overlay.


# CO2-layer sensitivity: LAB010 exclusion and repaired sampling signals

## Full natural theta, recalibrated CO2 layer, corrected SCCM

Comparisons reported here:

- **A — current:** full-theta calibration including LAB010, original signals;
- **B — sensitivity:** the same calibration excluding LAB010 from the natural fit, with the
  canonical sampling-repaired observed signals for LAB004/LAB007/LAB008;
- **A′ versus B isolation:** the previous no-LAB010 run on the original signals (frozen
  snapshot) versus the run above, so the repaired-signal effect is isolated from the LAB010
  exclusion.

The synthetic fit and cross-matrix machinery are retained unchanged. No kinetic theta is
re-estimated in this notebook.


## Estructura del modelo

Para la corrida (r), primero se corrige el cero instrumental en la señal nativa:

\[
b_r=\max\left[0,Q_{0.10}\{y_{sccm}(t):t\leq t_0+12\,h\}\right],\qquad
y_{corr}(t)=\max[y_{sccm}(t)-b_r,0]
\]

La conversión a g L⁻¹ h⁻¹, el filtrado de transientes y el suavizado ocurren después. Esta regla supone que el extremo inferior de las primeras 12 h representa cero instrumental; si ya existe flujo biológico durante toda esa ventana, puede sustraer parte de la señal real.

La primera muestra química que cumple pérdida de glucosa+fructosa ≥5 g/L o aumento de etanol ≥2 g/L marca el extremo superior del intervalo de activación. La muestra química anterior fija el extremo inferior. La fuente usa una rampa *smoothstep* continua:

\[
t_s=t_L+f_s(t_U-t_L),\quad
\Delta t=f_d(t_U-t_L),\quad
a_{chem}=3z^2-2z^3,\quad
z=\operatorname{clip}\!\left(\frac{t-t_s}{\Delta t},0,1\right)
\]

Los parámetros (f_s) y (f_d) son compartidos por matriz. Esto permite un aumento pequeño y gradual desde (t_s), pero mantiene el comienzo ligado al bracket químico en vez de introducir una latencia independiente por fermentación.

\[
\frac{dO_2}{dt}=-q_{O_2,max}X\frac{O_2}{K_{O_2}+O_2},\qquad
\phi_{ana}=\frac{K_{ana}^{h}}{K_{ana}^{h}+O_2^{h}}
\]

\[
q_{prod}=a_{chem}(t)\left[q_{bio}\left(f_{Crabtree}+(1-f_{Crabtree})\phi_{ana}\right)+q_{resp}\right]
\]

Para una adición en \(t_N\), la fracción utilizada es causal:

\[
f_N(t)=\operatorname{clip}\left(\frac{t-t_N}{t_{rise}},0,1\right),\qquad
q_{bio,N}=\left[1+(g_N-1)f_N(t)\right]q_{bio,0}
+f_N(t)\left(q_{bio,+N}-q_{bio,0}\right)
\]

La biomasa usa la misma diferencia con/sin pulso, pero sin \(g_N\). Así, el término de actividad puede capturar el aumento de capacidad de transporte descrito por el grupo de Sablayrolles sin imponer crecimiento ficticio.

\[
C^*_{CO_2}=s_{sat}\,1.69\,e^{-0.032(T-20)}e^{0.0016E}e^{-0.0012(G+F)}
\]

\[
\frac{dC_{CO_2}}{dt}=q_{prod}-q_{gas},\qquad
q_{gas}=\min\!\left[k_{release}C_{CO_2}
\left(f_0+(1-f_0)\frac{C_{CO_2}}{C_{CO_2}+C^*_{CO_2}}\right),
\frac{C_{CO_2}}{\Delta t}+q_{prod}\right],\quad f_0=0.05
\]

La formulación anterior, usada como comparador directo, tenía (q_{gas}=k_{release}\operatorname{softplus}(C_{CO_2}-C^*_{CO_2})): eso mantenía la predicción pegada a cero hasta acumular suficiente CO₂ disuelto. El nuevo piso (f_0) representa liberación sub-saturada pequeña y evita ese umbral duro.

El inicio observado se define como el primero de tres puntos horarios consecutivos sobre el mayor valor entre: límite de detección local y línea base inicial +10 % del rango dinámico. Además se reporta la primera emisión sostenida sobre 0.005 g L⁻¹ h⁻¹ y la duración visual 2–10 % del ascenso. El ajuste conserva el residuo de tiempo de inicio y, en lotes pulsados, el desfase del máximo durante las 72 h posteriores a la adición, además de los residuos del perfil completo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython import get_ipython
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.name != "pyomo-doe" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "pyomo-doe":
    raise RuntimeError("Execute this notebook from the repository or a descendant")

FM = ROOT / "fermentation_model"
for path in (FM, FM / "laboratory_2026", FM / "pilot_2025"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from laboratory_2026 import run_co2_matrix_cross_validation_2026_full_theta as analysis

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
print("Repository:", ROOT)
print("Current model:", analysis.MODEL_NAME)
print("Direct comparator:", analysis.THRESHOLD_RELEASE_MODEL_NAME)
print("Historical comparator:", analysis.LEGACY_MODEL_NAME)
print("Holdouts:", analysis.HOLDOUTS)
print("Excluded:", analysis.EXCLUDED_BATCHES)

# Sensitivity isolation: redirect every artifact to a new directory and alter only
# natural-fit membership. The original module and runner files remain untouched.
BASELINE_RESULTS_DIR = analysis.RESULTS_DIR
SENSITIVITY_RESULTS_DIR = (
    analysis.SCRIPT_DIR
    / "results"
    / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010"
)
analysis.RESULTS_DIR = SENSITIVITY_RESULTS_DIR
analysis.PLOT_DIR = SENSITIVITY_RESULTS_DIR / "figures"
analysis.NOTEBOOK_PATH = (
    analysis.NOTEBOOK_DIR
    / "co2_solubility_o2_cross_matrix_2026_full_theta_no_lab010.ipynb"
)
analysis.EXECUTED_NOTEBOOK_PATH = (
    analysis.NOTEBOOK_DIR
    / "co2_solubility_o2_cross_matrix_2026_full_theta_no_lab010.executed.ipynb"
)

SENSITIVITY_EXCLUDED_BATCH = "LAB010"
_historical = analysis.historical
_original_fit_matrix = _historical.fit_matrix
_original_calibration_batch_names = _historical._calibration_batch_names
_original_predict_and_score = _historical.predict_and_score

def _without_lab010_from_natural_fit(matrix, observations):
    adjusted = observations.copy()
    if matrix == "natural":
        adjusted.loc[adjusted["batch"].eq(SENSITIVITY_EXCLUDED_BATCH), "calibratable"] = False
    return adjusted

def _sensitivity_fit_matrix(matrix, observations, cache, *args, **kwargs):
    return _original_fit_matrix(
        matrix, _without_lab010_from_natural_fit(matrix, observations), cache, *args, **kwargs
    )

def _sensitivity_calibration_batch_names(matrix, observations):
    names = _original_calibration_batch_names(matrix, observations)
    if matrix == "natural":
        names = [name for name in names if name != SENSITIVITY_EXCLUDED_BATCH]
    return names

def _sensitivity_predict_and_score(fits, observations, cache, *args, **kwargs):
    predictions, metrics, validation = _original_predict_and_score(
        fits, observations, cache, *args, **kwargs
    )
    for frame in (predictions, metrics):
        diagnostic = (
            frame["calibration_matrix"].eq("natural")
            & frame["target_matrix"].eq("natural")
            & frame["batch"].eq(SENSITIVITY_EXCLUDED_BATCH)
        )
        frame.loc[diagnostic, "role"] = "excluded_fit_frozen_diagnostic"
    return predictions, metrics, validation

_historical.fit_matrix = _sensitivity_fit_matrix
_historical._calibration_batch_names = _sensitivity_calibration_batch_names
_historical.predict_and_score = _sensitivity_predict_and_score

print("Baseline results (read-only comparator):", BASELINE_RESULTS_DIR.relative_to(ROOT))
print("Sensitivity results:", SENSITIVITY_RESULTS_DIR.relative_to(ROOT))
print("Natural fit exclusion:", SENSITIVITY_EXCLUDED_BATCH)
print("Natural holdout retained:", analysis.HOLDOUTS["natural"])


## Aislamiento de señales reparadas — único cambio de datos observados

Para LAB004, LAB007 y LAB008 la señal observada de CO2 utilizada por la calibración se reemplaza
por la versión canónica reparada frente a artefactos de muestreo construida en el análisis de
variabilidad histórica/microleaks (`co2_observations_hourly_sampling_repaired.csv`,
`observation_signal_version = experimental_repaired_plus_current_smoothing`). No se reconstruye
ninguna señal en este notebook: se reutilizan exactamente las filas de ese artefacto. El resto de
la tabla de observaciones (LAB005/006/010/011/012 y toda la matriz sintética) permanece
byte-idéntico al input congelado SCCM-corrected, y se verifica por aserción.


In [ ]:
# Repaired-signal isolation: the only observed-data intervention of this variant.
# LAB004/007/008 swap their observed CO2 signal for the canonical sampling-repaired
# versions; every other row of the frozen SCCM-corrected table is verified identical.
import numpy as np
REPAIRED_RESULTS_DIR = (
    analysis.SCRIPT_DIR
    / "results"
    / "co2_historical_variability_microleaks_2026_sampling_repaired"
)
REPAIRED_OBSERVATIONS_PATH = REPAIRED_RESULTS_DIR / "co2_observations_hourly_sampling_repaired.csv"
SIGNAL_SWAP_BATCHES = ("LAB004", "LAB007", "LAB008")

CORRECTED_FROZEN_DIR = analysis.HISTORICAL_CORRECTED_RESULTS_DIR
_baseline_observations = pd.read_csv(CORRECTED_FROZEN_DIR / "co2_observations_hourly.csv")
_repaired_natural = pd.read_csv(REPAIRED_OBSERVATIONS_PATH)

def _assert_identical_frames(a, b, label):
    assert list(a.columns) == list(b.columns), f"{label}: column mismatch"
    assert len(a) == len(b), f"{label}: row mismatch {len(a)} vs {len(b)}"
    for column in a.columns:
        left, right = a[column], b[column]
        if pd.api.types.is_numeric_dtype(left) and pd.api.types.is_numeric_dtype(right):
            assert np.array_equal(
                left.to_numpy(dtype=float), right.to_numpy(dtype=float), equal_nan=True
            ), f"{label}: numeric mismatch in {column}"
        else:
            assert left.astype(str).equals(right.astype(str)), f"{label}: mismatch in {column}"

# Canonical-signal guards: the diagnostics must repair exactly these three labs.
_flagged = sorted(
    _repaired_natural.loc[
        _repaired_natural["sampling_repair_applied"].astype(bool), "batch"
    ].unique()
)
assert _flagged == list(SIGNAL_SWAP_BATCHES), f"unexpected repaired set: {_flagged}"
assert _repaired_natural["sccm_conversion"].astype(str).eq("SCCM_CORRECTED").all()
_version = _repaired_natural.groupby("batch")["observation_signal_version"].first()
assert (
    _version.reindex(SIGNAL_SWAP_BATCHES)
    .eq("experimental_repaired_plus_current_smoothing")
    .all()
)

# Untouched natural batches must reproduce the frozen table exactly.
for _lab in sorted(set(_repaired_natural["batch"].unique()) - set(SIGNAL_SWAP_BATCHES)):
    _assert_identical_frames(
        _baseline_observations[_baseline_observations["batch"].eq(_lab)]
        .sort_values("t_h")
        .reset_index(drop=True)[_baseline_observations.columns],
        _repaired_natural[_repaired_natural["batch"].eq(_lab)]
        .sort_values("t_h")
        .reset_index(drop=True)[_baseline_observations.columns],
        f"untouched batch {_lab}",
    )

# Swapped batches keep the identical hourly grid; only the signal chain changes.
_parts = []
swap_audit_rows = []
for _batch, _group in _baseline_observations.groupby("batch", sort=False):
    _group = _group.sort_values("t_h").reset_index(drop=True)
    if _batch in SIGNAL_SWAP_BATCHES:
        _new = (
            _repaired_natural[_repaired_natural["batch"].eq(_batch)]
            .sort_values("t_h")
            .reset_index(drop=True)
        )
        assert len(_new) == len(_group), f"{_batch}: hourly support changed"
        assert np.array_equal(
            _group["t_h"].to_numpy(dtype=float), _new["t_h"].to_numpy(dtype=float)
        ), f"{_batch}: time grid changed"
        _old_signal = _group["co2_rate_g_l_h"].to_numpy(dtype=float)
        _new_signal = _new["co2_rate_g_l_h"].to_numpy(dtype=float)
        _t_h = _group["t_h"].to_numpy(dtype=float)
        swap_audit_rows.append(
            {
                "batch": _batch,
                "n_rows": int(len(_new)),
                "n_changed_signal_rows": int((~np.isclose(_old_signal, _new_signal)).sum()),
                "max_abs_signal_change_g_l_h": float(np.abs(_old_signal - _new_signal).max()),
                "integral_old_g_l": float(np.trapz(_old_signal, _t_h)),
                "integral_new_g_l": float(np.trapz(_new_signal, _t_h)),
                "left_censored_old": int(_group["left_censored"].astype(bool).sum()),
                "left_censored_new": int(_new["left_censored"].astype(bool).sum()),
                "observation_signal_version": "experimental_repaired_plus_current_smoothing",
            }
        )
        _parts.append(_new)
    else:
        _parts.append(_group)
repaired_support_observations = pd.concat(_parts, ignore_index=True)[
    _baseline_observations.columns
]
assert len(repaired_support_observations) == len(_baseline_observations)
signal_swap_audit = pd.DataFrame(swap_audit_rows)

# Frozen inputs are copied untouched; only the observation table is replaced, and the
# analysis module is pointed at this input directory before run_analysis is called.
OBSERVATION_INPUT_DIR = SENSITIVITY_RESULTS_DIR / "observation_input_sampling_repaired"
OBSERVATION_INPUT_DIR.mkdir(parents=True, exist_ok=True)
repaired_support_observations.to_csv(
    OBSERVATION_INPUT_DIR / "co2_observations_hourly.csv", index=False
)
for _frozen_name in (
    "fit_parameters.csv",
    "fit_start_diagnostics.csv",
    "frozen_mask_audit.csv",
    "native_corrected_mask_diagnostic.csv",
):
    (OBSERVATION_INPUT_DIR / _frozen_name).write_bytes(
        (CORRECTED_FROZEN_DIR / _frozen_name).read_bytes()
    )
analysis.HISTORICAL_CORRECTED_RESULTS_DIR = OBSERVATION_INPUT_DIR

# The reproduction gate stays strict for model-side quantities (fits and driver cache are
# unchanged, so predicted curves must reproduce the frozen baseline). The observed column
# is expected to differ exactly where the three signals were replaced.
_original_reproduction_audit = analysis._reproduction_audit

def _reproduction_audit_model_side(predictions, observations):
    saved = pd.read_csv(CORRECTED_FROZEN_DIR / "prediction_rows.csv")
    keys = ["calibration_matrix", "target_matrix", "batch", "time_h"]
    merged_audit = saved.merge(
        predictions, on=keys, suffixes=("_saved", "_recomputed"), validate="one_to_one"
    )
    if len(merged_audit) != len(saved) or len(merged_audit) != len(predictions):
        raise AssertionError(
            "Historical SCCM-corrected prediction support was not reproduced"
        )
    rows = []
    for column in ("observed_g_l_h", "predicted_g_l_h", "raw_model_qgas_g_l_h"):
        difference = (
            pd.to_numeric(merged_audit[f"{column}_recomputed"], errors="raise")
            - pd.to_numeric(merged_audit[f"{column}_saved"], errors="raise")
        ).abs()
        rows.append(
            {
                "quantity": column,
                "n_rows": len(difference),
                "max_absolute_difference": float(difference.max()),
                "mean_absolute_difference": float(difference.mean()),
            }
        )
    audit = pd.DataFrame(rows)
    model_side = float(
        audit.loc[
            audit["quantity"].ne("observed_g_l_h"), "max_absolute_difference"
        ].max()
    )
    if model_side > 1e-9:
        raise AssertionError(
            "Historical SCCM-corrected model predictions were not reproduced numerically"
        )
    return audit

analysis._reproduction_audit = _reproduction_audit_model_side

print("Repaired-signal source:", REPAIRED_OBSERVATIONS_PATH.relative_to(ROOT))
print("Repaired batches:", ", ".join(SIGNAL_SWAP_BATCHES))
display(signal_swap_audit.round(6))


## Ejecución completa y partición experimental

In [ ]:
result = analysis.run_analysis(n_starts=5, max_nfev=300, seed=20260812)

# Correct the presentation-only inventory label. The underlying observations remain present
# so LAB010 can be evaluated with the fitted parameters frozen.
result["inventory"].loc[
    result["inventory"]["batch"].eq(SENSITIVITY_EXCLUDED_BATCH), "split"
] = "excluded_fit_frozen_diagnostic"
assert set(_historical._calibration_batch_names("natural", result["observations"])) == {
    "LAB004", "LAB005", "LAB006", "LAB007", "LAB008", "LAB011"
}
assert analysis.HOLDOUTS["natural"] == "LAB012"
print("Verified natural fitting batches:", _historical._calibration_batch_names("natural", result["observations"]))

display(result["exclusions"])
display(result["inventory"][[
    "matrix", "experiment_code", "batch", "lot", "calibratable", "split",
    "acquisition_channel", "sensor_id", "sensor_zero_offset_sccm",
    "n_co2_hourly", "chemistry_first_h", "chemistry_last_h", "co2_first_h",
    "co2_last_h", "co2_peak_g_l_h", "median_setpoint_c",
    "n_artifacts_replaced", "n_left_censored_hourly", "note"
]])
display(result["natural_nutrient_pulses"][[
    "batch", "calendar_t_h", "density_crossing_t_h", "density_bracket_width_h",
    "model_pulse_time_h", "timing_source", "amount_N_kg_m3",
    "calendar_product", "calendar_dose", "excluded_from_co2_analysis"
]])
fig = analysis.plot_process_timeline_alignment(
    result["inventory"], result["nutrient_pulses"], save=False
)
plt.show()

La separación se hace por fermentación completa. Los pulsos LAB aportan 0.14 kg N m⁻³; el pulso inicial se ignora porque el YAN inicial ya está en t=0. En todos los LAB con cruce disponible, el tiempo del pulso se obtiene por interpolación del cruce de densidad 1040 g/L. Un bracket químico >24 h se conserva, pero se etiqueta como incertidumbre de tiempo. En DOE-F0X se usa el tiempo de proceso existente, sin recalcularlo por densidad ni calendario.

## Filtrado y sensibilidad del sensor

In [ ]:
display(result["qc_summary"][[
    "matrix", "experiment_code", "batch", "acquisition_channel", "sensor_id",
    "sensor_zero_offset_sccm", "sensor_zero_offset_g_l_h", "median_setpoint_c",
    "n_artifacts_replaced", "n_sampling_window_transients",
    "n_other_short_transients", "negative_signed_fraction",
    "n_left_censored_hourly", "n_cold_low_sensitivity_hourly",
    "smoothing_roughness_ratio", "smoothing_integral_ratio", "smoothing_peak_ratio"
]])
display(result["sensor_zero_offsets"])
fig = analysis.plot_sensor_zero_correction(
    result["sensor_qc"], result["sensor_zero_offsets"], save=False
)
plt.show()
fig = analysis.plot_data_overview(
    result["observations"], result["nutrient_pulses"], save=False
)
plt.show()
fig = analysis.plot_sensor_filter_examples(
    result["sensor_qc"], result["sampling_schedule"],
    result["nutrient_pulses"], save=False
)
plt.show()

El primer gráfico comprueba la corrección de cero antes de cualquier reconstrucción. El offset se estima por corrida porque los canales muestran deriva entre campañas. Luego, las excursiones asociadas a muestreo se reconstruyen sólo cuando el evento completo está respaldado por niveles pre/post consistentes; se aplica mediana robusta de 3 h y Savitzky–Golay cuadrático de 5 h. Los puntos bajo el umbral local aportan una penalización unilateral si el modelo excede el límite.

## Temperatura y evidencia química de activación

In [ ]:
display(result["driver_diagnostics"][[
    "matrix", "batch", "temperature_used_min_c", "temperature_used_mean_c",
    "temperature_used_max_c", "chemical_activity_lower_h",
    "chemical_activity_upper_h", "chemical_activity_center_h",
    "initial_o2_saturation_base_mg_l", "base_qprod_peak_g_l_h",
    "csat_base_min_g_l", "csat_base_max_g_l"
]])
display(result["temperature_alignment"])
fig = analysis.plot_temperature_profiles(
    result["observations"], result["temperature_inputs"],
    result["nutrient_pulses"], save=False
)
plt.show()

La temperatura medida/reconstruida entra tanto a los drivers cinéticos como a la solubilidad de CO₂; el setpoint queda como referencia. La tabla permite verificar, lote por lote, el intervalo químico que condiciona el encendido de la fuente.

## Auditoría obligatoria de theta y SCCM

Esta celda se ejecuta después de construir los drivers; si cualquiera de los tres CSV no contiene
exactamente los 17 parámetros esperados, el runner falla antes de la primera simulación. La tabla de
reconstrucción identifica las seis filas que históricamente provenían de `DEFAULT_THETA`.

In [ ]:
display(result["theta_validation_summary"])
display(result["theta_historical_audit"])
print("SCCM_CORRECTED [g/L/h/SCCM]:", result["manifest"]["sccm_conversion"]["factor_g_l_h_per_sccm_at_2L"])
print("Historical baseline reproduced:")
display(result["historical_reproduction_audit"])

## Comparación explícita A/B/C

A→B cuantifica solo el reemplazo del theta. B→C cuantifica la readaptación de la capa CO₂. A→C es
el efecto final. La descomposición escala/forma ajusta únicamente un multiplicador diagnóstico entre
dos curvas ya calculadas; no modifica B ni constituye un refit. Un valor alto de la fracción explicada
por escala indica cambio mayormente multiplicativo.

In [ ]:
display(result["state_objectives"])
display(result["state_parameter_changes"])
display(result["matrix_gain_compensation"])
display(result["state_validation"])
display(
    result["pairwise_shape_scale"]
    .query("transition == 'A_to_B'")
    .sort_values("direct_prediction_rmse_g_l_h", ascending=False)
)
display(result["identifiability_state_comparison"])
fig = analysis.plot_three_state_overlays(result["state_predictions"], save=False)
plt.show()
fig = analysis.plot_matrix_gain_three_states(result["state_parameters"], save=False)
plt.show()

## Estado C — recalibración CO₂ por matriz con theta completo congelado

In [ ]:
display(result["fit_parameters"])
best_starts = (
    result["fit_starts"].sort_values("wsse_equal_batch")
    .groupby("matrix", as_index=False).first()
)
display(best_starts[[
    "matrix", "success", "nfev", "wsse_equal_batch", "kCO2_release_h",
    "CO2sat_scale", "O2_qmax_mg_gdw_h", "O2_initial_scale",
    "pulse_t_rise_h", "pulse_activity_gain",
    "chem_activation_start_fraction", "chem_activation_duration_fraction",
    "matrix_gain"
]])
fig = analysis.plot_parameter_comparison(result["fit_parameters"], save=False)
plt.show()
for matrix in ("synthetic", "natural"):
    fig = analysis.plot_calibration_overlays(result["predictions"], matrix, save=False)
    plt.show()

`active_bound=True` indica que el dato sólo acota el parámetro en el borde permitido. Debe leerse como tensión estructural o falta de identificabilidad, no como una estimación interior resuelta. `chem_activation_start_fraction` ubica el inicio dentro del bracket químico y `chem_activation_duration_fraction` escala la duración por el ancho de ese bracket. `pulse_activity_gain=1` significa ausencia de modulación sobre la actividad preexistente; >1 es boost y <1 es atenuación.

En esta sección todos los estimates corresponden al estado C. Los estados A y B comparten exactamente los parámetros CO₂ históricos; B no ejecuta fitting.

## Estimabilidad de inicio y duración

In [ ]:
display(result["jacobian_identifiability"])
display(
    result["local_parameter_correlations"].query(
        "parameter_1 in ['chem_activation_start_fraction', 'chem_activation_duration_fraction']"
    )[["matrix", "parameter_1", "parameter_2", "local_log_parameter_correlation"]]
    .sort_values(["matrix", "parameter_1", "local_log_parameter_correlation"])
)
display(result["loo_activation_summary"])
display(result["loo_activation_estimates"][[
    "matrix", "omitted_batch", "success", "nfev",
    "chem_activation_start_fraction", "chem_activation_duration_fraction"
]])
display(result["activation_profiles"])
fig = analysis.plot_activation_identifiability(
    result["activation_profiles"], result["loo_activation_summary"], save=False
)
plt.show()

El Jacobiano evalúa sensibilidad local en log-parámetros; un número de condición alto indica direcciones compensables. Los perfiles fijan uno de los dos parámetros y reoptimizan todos los demás. La franja naranjo muestra cuánto cambia la estimación al retirar una fermentación completa. La línea horizontal de 5 % es sólo un umbral descriptivo de sensibilidad del objetivo, no un intervalo de confianza de verosimilitud.

La identificabilidad se compara entre A y C, ambos óptimos de calibración. No se atribuye identificabilidad a B porque deliberadamente no es un óptimo.

## ¿Se corrigió la cola inicial?

In [ ]:
native_transition = result["source_transition_diagnostics"].query("native_matrix_fit").copy()
display(native_transition[[
    "target_matrix", "experiment_code", "batch", "median_setpoint_c",
    "cold_operation", "chemical_activity_lower_h", "chemical_activity_upper_h",
    "source_10pct_peak_onset_h", "o2_below_anaerobic_halfpoint_h",
    "observed_onset_h", "predicted_onset_h", "onset_delay_h",
    "O2_qmax_mg_gdw_h", "O2_initial_scale", "n_pulse_time_h",
    "pulse_t_rise_h", "pulse_activity_gain",
    "chem_activation_start_fraction", "chem_activation_duration_fraction",
    "pulse_utilization_complete_h",
    "observed_postpulse_peak_h", "predicted_postpulse_peak_h",
    "postpulse_peak_delay_h"
]])
fig = analysis.plot_initial_release_comparison(
    result["predictions"], result["previous_predictions"],
    result["batch_metrics"], result["previous_batch_metrics"], save=False
)
plt.show()
fig = analysis.plot_onset_model_comparison(
    result["batch_metrics"], result["previous_batch_metrics"], save=False
)
plt.show()
fig = analysis.plot_pulse_response_comparison(
    result["predictions"], result["previous_predictions"],
    result["batch_metrics"], result["previous_batch_metrics"],
    result["source_transition_diagnostics"], save=False
)
plt.show()

El primer gráfico es el control visual principal: amplía el inicio de cada lote y compara liberación por umbral (naranjo) con liberación continua (azul). La línea horizontal marca 0.005 g L⁻¹ h⁻¹; el texto cuantifica la primera emisión y la duración 2–10 %. El segundo resume el error de inicio. El tercero amplía cada lote pulsado: línea magenta = adición; línea morada = término de la disponibilidad gradual estimada.

## Validación retenida y transferencia cruzada

In [ ]:
validation_columns = [
    "scenario", "calibration_matrix", "target_matrix", "experiment_code", "batch",
    "n", "n_total", "n_left_censored", "rmse_g_l_h", "nrmse_peak", "bias_g_l_h",
    "correlation", "r2", "integral_ratio_pred_over_observed_lower_bound",
    "observed_onset_h", "predicted_onset_h", "onset_delay_h",
    "predicted_first_emission_0p005_h", "observed_visual_rise_duration_h",
    "predicted_visual_rise_duration_h", "visual_rise_duration_error_h",
    "pulse_time_h", "observed_postpulse_peak_h", "predicted_postpulse_peak_h",
    "postpulse_peak_delay_h"
]
display(result["validation"][validation_columns])
display(result["model_comparison"])
display(result["filter_impact"])
fig = analysis.plot_validation(result["predictions"], result["validation"], save=False)
plt.show()
fig = analysis.plot_validation_model_comparison(
    result["predictions"], result["previous_predictions"],
    result["validation"], result["previous_validation"], save=False
)
plt.show()

`model_comparison` usa exactamente los mismos holdouts para la liberación por umbral anterior y la liberación continua nueva, conservando la misma respuesta finita al pulso. Valores negativos en las columnas `*_change_continuous_minus_threshold_release` significan mejora. `filter_impact` compara los ajustes crudo y filtrado sobre el mismo soporte cuantificable.

## Diagnóstico por fermentación

In [ ]:
display(result["batch_metrics"][[
    "calibration_matrix", "target_matrix", "experiment_code", "batch", "role",
    "n", "n_left_censored", "rmse_g_l_h", "nrmse_peak", "bias_g_l_h",
    "correlation", "integral_ratio_pred_over_observed_lower_bound",
    "observed_onset_h", "predicted_onset_h", "onset_delay_h", "pulse_time_h",
    "predicted_first_emission_0p005_h", "observed_visual_rise_duration_h",
    "predicted_visual_rise_duration_h", "visual_rise_duration_error_h",
    "observed_postpulse_peak_h", "predicted_postpulse_peak_h",
    "postpulse_peak_delay_h"
]].sort_values(["calibration_matrix", "target_matrix", "batch"]))

## Experimento pendiente para distinguir sensor de biología

El umbral dependiente de temperatura sigue siendo operacional, no un LOD/LOQ certificado. Para identificarlo se requiere un ensayo a 15, 18 y 21 °C con el mismo medio y volumen, referencia independiente de CO₂ y escalones de 0.01–0.30 g CO₂ L⁻¹ h⁻¹. En cada temperatura deben estimarse LOD, LOQ, sesgo, repetibilidad y tiempo de respuesta.

La corrección por percentil inicial también es operacional. Debe reemplazarse por blancos de gas sin fermentación medidos al comienzo y al final de cada corrida en cada canal. Eso separaría offset, deriva y flujo biológico temprano sin depender de una ventana temporal elegida.

La validación biológica complementaria debe iniciar con medición química frecuente durante las primeras 48–72 h, especialmente en frío, para estrechar el intervalo de activación. Sin ese muestreo, la química sólo identifica un intervalo y no un instante exacto.

## Resumen reproducible

In [ ]:
print("SCCM_CORRECTED:", result["manifest"]["sccm_conversion"]["factor_g_l_h_per_sccm_at_2L"], "g/L/h/SCCM")
display(result["state_objectives"])
print()
print("matrix_gain (parámetro empírico de observación):")
display(result["matrix_gain_compensation"])

direct = result["pairwise_shape_scale"].query("transition == 'A_to_B'")
native = direct.query("calibration_matrix == target_matrix")
print()
print("Batches más sensibles a A→B (fits nativos):")
display(native.nlargest(8, "direct_prediction_rmse_g_l_h")[[
    "calibration_matrix", "target_matrix", "experiment_code", "batch", "role",
    "direct_prediction_rmse_g_l_h", "best_multiplicative_scale_right_from_left",
    "shape_rmse_after_best_scale_g_l_h", "fraction_squared_change_explained_by_scale",
    "integral_ratio_right_over_left", "peak_ratio_right_over_left", "peak_time_change_h"
]])

print()
print("Holdouts y transferencia A/B/C:")
display(result["state_validation"])
print()
print("Parámetros en límite por estado:")
display(result["state_parameters"].query("active_bound"))
print()
print("Identificabilidad local A vs C:")
display(result["identifiability_state_comparison"])
print()
print("Artefactos:", analysis.RESULTS_DIR.relative_to(ROOT))

## Artifacts

The complete recalibration and sensitivity suite is written only to
`results/co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010/`.
The current full-theta calibration folder is used read-only as scenario A. In scenario B,
LAB010 is excluded from the natural objective, objective profiles and leave-one-batch-out
refits; LAB012 remains the internal holdout.


## LAB010 influence sensitivity — current A versus no-LAB010 B

The current full-theta fit is loaded from its existing artifacts as **A** (original signals,
LAB010 included). The result generated above is **B**: LAB010 excluded and the repaired
sampling signals for LAB004/007/008, so this section still bundles both interventions. The
repaired-signal-only isolation against the previous no-LAB010 run is reported in the dedicated
section below. Integrals are recomputed from the stored prediction rows over each complete
hourly support; the remaining statistics are the established metrics from the unchanged
scoring function.


In [ ]:
import json
from dataclasses import replace
import numpy as np

CURRENT_DIR = BASELINE_RESULTS_DIR
CURRENT_FIT_PARAMETERS_PATH = CURRENT_DIR / "fit_parameters.csv"
CURRENT_FIT_STARTS_PATH = CURRENT_DIR / "fit_start_diagnostics.csv"
CURRENT_METRICS_PATH = CURRENT_DIR / "batch_metrics.csv"
CURRENT_PREDICTIONS_PATH = CURRENT_DIR / "prediction_rows.csv"

current_parameters = pd.read_csv(CURRENT_FIT_PARAMETERS_PATH)
current_fit_starts = pd.read_csv(CURRENT_FIT_STARTS_PATH)
current_metrics = pd.read_csv(CURRENT_METRICS_PATH)
current_predictions = pd.read_csv(CURRENT_PREDICTIONS_PATH)

def _native_natural(frame):
    return frame[
        frame["calibration_matrix"].eq("natural")
        & frame["target_matrix"].eq("natural")
    ].copy()

def _integrals(predictions):
    rows = []
    for batch, group in _native_natural(predictions).groupby("batch", sort=True):
        group = group.sort_values("time_h")
        rows.append({
            "batch": batch,
            "observed_integral_g_l": float(np.trapz(group["observed_g_l_h"], group["time_h"])),
            "predicted_integral_g_l": float(np.trapz(group["predicted_g_l_h"], group["time_h"])),
        })
    return pd.DataFrame(rows)

def _scenario_metrics(label, metrics, predictions):
    selected = _native_natural(metrics).merge(_integrals(predictions), on="batch", validate="one_to_one")
    selected.insert(0, "scenario", label)
    selected["integral_ratio_recomputed"] = (
        selected["predicted_integral_g_l"] / selected["observed_integral_g_l"].clip(lower=1e-12)
    )
    return selected

metrics_a = _scenario_metrics("A_current_with_LAB010", current_metrics, current_predictions)
metrics_b = _scenario_metrics("B_no_LAB010", result["batch_metrics"], result["predictions"])
sensitivity_metrics_long = pd.concat([metrics_a, metrics_b], ignore_index=True)

metric_names = [
    "rmse_g_l_h", "nrmse_peak", "bias_g_l_h", "correlation", "r2",
    "observed_integral_g_l", "predicted_integral_g_l", "integral_ratio_recomputed",
    "observed_peak_g_l_h", "predicted_peak_g_l_h", "observed_peak_time_h",
    "predicted_peak_time_h", "observed_onset_h", "predicted_onset_h", "onset_delay_h",
]
wide_a = metrics_a[["batch", *metric_names]].copy()
wide_b = metrics_b[["batch", *metric_names]].copy()
sensitivity_metric_changes = wide_a.merge(
    wide_b, on="batch", suffixes=("_A", "_B"), validate="one_to_one"
)
for metric in metric_names:
    sensitivity_metric_changes[f"{metric}_change_B_minus_A"] = (
        sensitivity_metric_changes[f"{metric}_B"] - sensitivity_metric_changes[f"{metric}_A"]
    )
    sensitivity_metric_changes[f"{metric}_pct_change_B_vs_A"] = 100.0 * (
        sensitivity_metric_changes[f"{metric}_B"] - sensitivity_metric_changes[f"{metric}_A"]
    ) / sensitivity_metric_changes[f"{metric}_A"].abs().replace(0.0, np.nan)

parameters_a = current_parameters.copy()
parameters_a.insert(0, "scenario", "A_current_with_LAB010")
parameters_b = result["fit_parameters"].copy()
parameters_b.insert(0, "scenario", "B_no_LAB010")
sensitivity_parameters_long = pd.concat([parameters_a, parameters_b], ignore_index=True)
sensitivity_parameter_changes = current_parameters.merge(
    result["fit_parameters"], on=["calibration_matrix", "parameter"],
    suffixes=("_A", "_B"), validate="one_to_one"
)
sensitivity_parameter_changes["absolute_change_B_minus_A"] = (
    sensitivity_parameter_changes["estimate_B"] - sensitivity_parameter_changes["estimate_A"]
)
sensitivity_parameter_changes["percent_change_B_vs_A"] = 100.0 * (
    sensitivity_parameter_changes["absolute_change_B_minus_A"]
    / sensitivity_parameter_changes["estimate_A"].abs().replace(0.0, np.nan)
)

fit_batches_a = ["LAB004", "LAB005", "LAB006", "LAB007", "LAB008", "LAB010", "LAB011"]
fit_batches_b = _historical._calibration_batch_names("natural", result["observations"])
objective_a = (
    current_fit_starts.sort_values("wsse_equal_batch")
    .groupby("matrix", as_index=False).first()[["matrix", "wsse_equal_batch"]]
)
objective_a.insert(0, "scenario", "A_current_with_LAB010")
objective_a["support"] = "native_fit_support"
objective_b = pd.DataFrame([
    {"scenario": "B_no_LAB010", "matrix": matrix, "wsse_equal_batch": fit["wsse_equal_batch"]}
    for matrix, fit in result["fits"].items()
])
objective_b["support"] = "native_fit_support"

# Fair WSSE comparison on the identical six-batch natural support used by B.
_common_theta_sets, _, _ = analysis.load_complete_theta_sets()
_common_batches, _, _ = _historical.load_batches()
_common_cache, _ = _historical.build_driver_cache(
    _common_batches,
    {"natural": _common_theta_sets["natural"], "synthetic": _common_theta_sets["synthetic"]},
    result["observations"],
)
_current_natural_parameters = dict(zip(
    current_parameters.loc[current_parameters["calibration_matrix"].eq("natural"), "parameter"],
    current_parameters.loc[current_parameters["calibration_matrix"].eq("natural"), "estimate"],
))
_current_natural_parameters["model"] = analysis.MODEL_NAME
_common_residual_a = analysis._fixed_parameter_residual(
    _current_natural_parameters, fit_batches_b, result["observations"], _common_cache
)
common_support_objectives = pd.DataFrame([
    {"scenario": "A_current_with_LAB010", "matrix": "natural",
     "wsse_equal_batch": float(np.dot(_common_residual_a, _common_residual_a)),
     "support": "common_six_batch_support"},
    {"scenario": "B_no_LAB010", "matrix": "natural",
     "wsse_equal_batch": float(result["fits"]["natural"]["wsse_equal_batch"]),
     "support": "common_six_batch_support"},
])
sensitivity_objectives = pd.concat([objective_a, objective_b, common_support_objectives], ignore_index=True)

sensitivity_inventory = pd.DataFrame([
    {"scenario": "A_current_with_LAB010", "natural_fit_batches": ", ".join(fit_batches_a),
     "n_natural_fit_batches": len(fit_batches_a), "natural_internal_holdout": "LAB012",
     "LAB010_role": "calibration"},
    {"scenario": "B_no_LAB010", "natural_fit_batches": ", ".join(fit_batches_b),
     "n_natural_fit_batches": len(fit_batches_b), "natural_internal_holdout": "LAB012",
     "LAB010_role": "excluded fit; frozen diagnostic"},
])

print("Calibration membership audit")
display(sensitivity_inventory)
print("Objective comparison")
display(sensitivity_objectives.round(6))
print("CO2-layer parameters A/B")
display(sensitivity_parameter_changes.round(6))
print("LAB010 frozen diagnostic and requested validation batches")
display(sensitivity_metrics_long[
    sensitivity_metrics_long["batch"].isin(["LAB010", "LAB011", "LAB012"])
][["scenario", "batch", "role", *metric_names]].round(6))


## Identifiability sensitivity

The local log-Jacobian, parameter-correlation matrix, activation profiles and LOO refits for B
all use the six-batch natural fit. The corresponding current-calibration artifacts are loaded as A.


In [ ]:
current_jacobian = pd.read_csv(CURRENT_DIR / "activation_jacobian_identifiability.csv")
current_correlations = pd.read_csv(CURRENT_DIR / "activation_local_parameter_correlations.csv")
current_profiles = pd.read_csv(CURRENT_DIR / "activation_objective_profiles.csv")
current_loo_summary = pd.read_csv(CURRENT_DIR / "activation_leave_one_batch_out_summary.csv")
current_loo_estimates = pd.read_csv(CURRENT_DIR / "activation_leave_one_batch_out_estimates.csv")

def _tag(frame, scenario):
    tagged = frame.copy()
    tagged.insert(0, "scenario", scenario)
    return tagged

sensitivity_identifiability = pd.concat([
    _tag(current_jacobian, "A_current_with_LAB010"),
    _tag(result["jacobian_identifiability"], "B_no_LAB010"),
], ignore_index=True)
sensitivity_correlations = pd.concat([
    _tag(current_correlations, "A_current_with_LAB010"),
    _tag(result["local_parameter_correlations"], "B_no_LAB010"),
], ignore_index=True)
sensitivity_profiles = pd.concat([
    _tag(current_profiles, "A_current_with_LAB010"),
    _tag(result["activation_profiles"], "B_no_LAB010"),
], ignore_index=True)
sensitivity_loo_summary = pd.concat([
    _tag(current_loo_summary, "A_current_with_LAB010"),
    _tag(result["loo_activation_summary"], "B_no_LAB010"),
], ignore_index=True)
sensitivity_loo_estimates = pd.concat([
    _tag(current_loo_estimates, "A_current_with_LAB010"),
    _tag(result["loo_activation_estimates"], "B_no_LAB010"),
], ignore_index=True)

print("Local Jacobian condition/rank")
display(sensitivity_identifiability.round(6))
print("Activation-parameter LOO stability")
display(sensitivity_loo_summary.round(6))
print("Largest absolute off-diagonal local correlations — natural matrix")
display(
    sensitivity_correlations[
        sensitivity_correlations["matrix"].eq("natural")
        & sensitivity_correlations["parameter_1"].ne(sensitivity_correlations["parameter_2"])
    ].assign(abs_correlation=lambda d: d["local_log_parameter_correlation"].abs())
    .sort_values(["scenario", "abs_correlation"], ascending=[True, False])
    .groupby("scenario", as_index=False).head(8)
    .round(6)
)


## Repaired-signal isolation — previous-signals A′ versus repaired-signals B (both no-LAB010)

**A′** is the frozen snapshot of the previous execution of this same no-LAB010 experiment with the
original LAB004/007/008 signals (`..._no_lab010_previous_signals_A`). **B** is the run above with
the repaired signals. Fit membership, theta, SCCM factor, objective, bounds, weights, censoring
treatment, multistart and optimizer are identical; the only difference is the observed signal of
those three batches. The cross-objective table additionally evaluates each parameter set on both
observation supports to attribute the WSSE change to the signal replacement.


In [ ]:
import math

PREVIOUS_SIGNALS_DIR = (
    analysis.SCRIPT_DIR
    / "results"
    / "co2_matrix_cross_validation_2026_full_theta_sccm_corrected_no_lab010_previous_signals_A"
)
previous_parameters = pd.read_csv(PREVIOUS_SIGNALS_DIR / "fit_parameters.csv")
previous_fit_starts = pd.read_csv(PREVIOUS_SIGNALS_DIR / "fit_start_diagnostics.csv")
previous_metrics = pd.read_csv(PREVIOUS_SIGNALS_DIR / "batch_metrics.csv")
previous_predictions = pd.read_csv(PREVIOUS_SIGNALS_DIR / "prediction_rows.csv")
previous_jacobian = pd.read_csv(
    PREVIOUS_SIGNALS_DIR / "activation_jacobian_identifiability.csv"
)
previous_correlations = pd.read_csv(
    PREVIOUS_SIGNALS_DIR / "activation_local_parameter_correlations.csv"
)
previous_loo_summary = pd.read_csv(
    PREVIOUS_SIGNALS_DIR / "activation_leave_one_batch_out_summary.csv"
)
previous_loo_estimates = pd.read_csv(
    PREVIOUS_SIGNALS_DIR / "activation_leave_one_batch_out_estimates.csv"
)

# Guard: the snapshot is exactly the previous no-LAB010 run on the original signals.
_previous_observations = pd.read_csv(PREVIOUS_SIGNALS_DIR / "co2_observations_hourly.csv")
_swapped_rows = _previous_observations["batch"].isin(SIGNAL_SWAP_BATCHES).to_numpy()
_assert_identical_frames(
    _previous_observations.loc[~_swapped_rows].reset_index(drop=True)[
        _baseline_observations.columns
    ],
    repaired_support_observations.loc[~_swapped_rows].reset_index(drop=True)[
        _baseline_observations.columns
    ],
    "non-swapped rows",
)
assert (
    _previous_observations.loc[_swapped_rows, "co2_rate_g_l_h"]
    .to_numpy(dtype=float)
    .tolist()
    != repaired_support_observations.loc[_swapped_rows, "co2_rate_g_l_h"]
    .to_numpy(dtype=float)
    .tolist()
)

def _tag_signal(frame, scenario):
    tagged = frame.copy()
    tagged.insert(0, "scenario", scenario)
    return tagged

# --- parameters -----------------------------------------------------------
signal_parameter_changes = previous_parameters.merge(
    result["fit_parameters"],
    on=["calibration_matrix", "parameter"],
    suffixes=("_A_previous_signals", "_B_repaired_signals"),
    validate="one_to_one",
)
signal_parameter_changes["absolute_change_B_minus_A"] = (
    signal_parameter_changes["estimate_B_repaired_signals"]
    - signal_parameter_changes["estimate_A_previous_signals"]
)
signal_parameter_changes["percent_change_B_vs_A"] = 100.0 * (
    signal_parameter_changes["absolute_change_B_minus_A"]
    / signal_parameter_changes["estimate_A_previous_signals"].abs().replace(0.0, np.nan)
)
signal_parameters_long = pd.concat(
    [
        _tag_signal(previous_parameters, "A_previous_signals"),
        _tag_signal(result["fit_parameters"], "B_repaired_signals"),
    ],
    ignore_index=True,
)

# --- per-batch metrics ----------------------------------------------------
metrics_prev_signals = _scenario_metrics("A_previous_signals", previous_metrics, previous_predictions)
metrics_repaired_signals = _scenario_metrics(
    "B_repaired_signals", result["batch_metrics"], result["predictions"]
)
signal_metrics_long = pd.concat(
    [metrics_prev_signals, metrics_repaired_signals], ignore_index=True
)
wide_prev_signals = metrics_prev_signals[["batch", *metric_names]].copy()
wide_repaired_signals = metrics_repaired_signals[["batch", *metric_names]].copy()
signal_metric_changes = wide_prev_signals.merge(
    wide_repaired_signals,
    on="batch",
    suffixes=("_A_previous_signals", "_B_repaired_signals"),
    validate="one_to_one",
)
for metric in metric_names:
    signal_metric_changes[f"{metric}_change_B_minus_A"] = (
        signal_metric_changes[f"{metric}_B_repaired_signals"]
        - signal_metric_changes[f"{metric}_A_previous_signals"]
    )

# --- WSSE attribution on the identical six-batch support ------------------
_previous_natural = previous_parameters[previous_parameters["calibration_matrix"].eq("natural")]
_previous_natural_parameters = dict(zip(_previous_natural["parameter"], _previous_natural["estimate"]))
_previous_natural_parameters["model"] = analysis.MODEL_NAME
_repaired_natural_table = result["fit_parameters"][
    result["fit_parameters"]["calibration_matrix"].eq("natural")
]
_repaired_natural_parameters = dict(
    zip(_repaired_natural_table["parameter"], _repaired_natural_table["estimate"])
)
_repaired_natural_parameters["model"] = analysis.MODEL_NAME

_res_A_on_A = analysis._fixed_parameter_residual(
    _previous_natural_parameters, fit_batches_b, _previous_observations, _common_cache
)
_res_A_on_B = analysis._fixed_parameter_residual(
    _previous_natural_parameters, fit_batches_b, result["observations"], _common_cache
)
_res_B_on_A = analysis._fixed_parameter_residual(
    _repaired_natural_parameters, fit_batches_b, _previous_observations, _common_cache
)
_res_B_on_B = analysis._fixed_parameter_residual(
    _repaired_natural_parameters, fit_batches_b, result["observations"], _common_cache
)
_wsse = lambda r: float(np.dot(r, r))
signal_objectives = pd.DataFrame(
    [
        {
            "parameter_set": "A_previous_signals",
            "observation_support": "A_previous_signals",
            "wsse_equal_batch": _wsse(_res_A_on_A),
        },
        {
            "parameter_set": "A_previous_signals",
            "observation_support": "B_repaired_signals",
            "wsse_equal_batch": _wsse(_res_A_on_B),
        },
        {
            "parameter_set": "B_repaired_signals",
            "observation_support": "A_previous_signals",
            "wsse_equal_batch": _wsse(_res_B_on_A),
        },
        {
            "parameter_set": "B_repaired_signals",
            "observation_support": "B_repaired_signals",
            "wsse_equal_batch": _wsse(_res_B_on_B),
        },
    ]
)
_prev_fit_wsse = float(
    previous_fit_starts.sort_values("wsse_equal_batch").query("matrix == 'natural'")
    .iloc[0]["wsse_equal_batch"]
)
assert math.isclose(
    signal_objectives.loc[0, "wsse_equal_batch"], _prev_fit_wsse, rel_tol=1e-6
), (signal_objectives.loc[0, "wsse_equal_batch"], _prev_fit_wsse)
assert math.isclose(
    signal_objectives.loc[3, "wsse_equal_batch"],
    float(result["fits"]["natural"]["wsse_equal_batch"]),
    rel_tol=1e-6,
)

# --- identifiability, correlations, LOO -----------------------------------
signal_identifiability = pd.concat(
    [
        _tag_signal(previous_jacobian, "A_previous_signals"),
        _tag_signal(result["jacobian_identifiability"], "B_repaired_signals"),
    ],
    ignore_index=True,
)
signal_correlations = pd.concat(
    [
        _tag_signal(previous_correlations, "A_previous_signals"),
        _tag_signal(result["local_parameter_correlations"], "B_repaired_signals"),
    ],
    ignore_index=True,
)
signal_loo_summary = pd.concat(
    [
        _tag_signal(previous_loo_summary, "A_previous_signals"),
        _tag_signal(result["loo_activation_summary"], "B_repaired_signals"),
    ],
    ignore_index=True,
)
signal_loo_estimates = pd.concat(
    [
        _tag_signal(previous_loo_estimates, "A_previous_signals"),
        _tag_signal(result["loo_activation_estimates"], "B_repaired_signals"),
    ],
    ignore_index=True,
)

print("CO2-layer parameters — natural matrix, previous versus repaired signals")
display(
    signal_parameter_changes[
        signal_parameter_changes["calibration_matrix"].eq("natural")
    ].round(6)
)
print("WSSE attribution on the identical six-batch natural support")
display(signal_objectives.round(6))
print("Repaired batches — observed-side metric changes carried by the new signals")
display(
    signal_metric_changes[signal_metric_changes["batch"].isin(SIGNAL_SWAP_BATCHES)][
        [
            "batch",
            "observed_integral_g_l_A_previous_signals",
            "observed_integral_g_l_B_repaired_signals",
            "observed_peak_g_l_h_A_previous_signals",
            "observed_peak_g_l_h_B_repaired_signals",
            "observed_peak_time_h_A_previous_signals",
            "observed_peak_time_h_B_repaired_signals",
            "observed_onset_h_A_previous_signals",
            "observed_onset_h_B_repaired_signals",
        ]
    ].round(6)
)
print("Repaired batches — full metric changes (same frozen model form, refit parameters)")
display(
    signal_metric_changes[signal_metric_changes["batch"].isin(SIGNAL_SWAP_BATCHES)][
        ["batch", "rmse_g_l_h_A_previous_signals", "rmse_g_l_h_B_repaired_signals",
         "rmse_g_l_h_change_B_minus_A", "nrmse_peak_A_previous_signals",
         "nrmse_peak_B_repaired_signals", "bias_g_l_h_A_previous_signals",
         "bias_g_l_h_B_repaired_signals", "r2_A_previous_signals", "r2_B_repaired_signals"]
    ].round(6)
)
signal_metric_changes["role_B_repaired_signals"] = signal_metric_changes["batch"].map(
    metrics_repaired_signals.set_index("batch")["role"]
)
print("LAB011 and LAB012 — unchanged signals, effect of the refit alone")
display(
    signal_metric_changes[signal_metric_changes["batch"].isin(["LAB011", "LAB012"])][
        ["batch", "role_B_repaired_signals", "rmse_g_l_h_A_previous_signals",
         "rmse_g_l_h_B_repaired_signals", "rmse_g_l_h_change_B_minus_A",
         "bias_g_l_h_A_previous_signals", "bias_g_l_h_B_repaired_signals",
         "r2_A_previous_signals", "r2_B_repaired_signals",
         "integral_ratio_recomputed_A_previous_signals",
         "integral_ratio_recomputed_B_repaired_signals"]
    ].round(6)
)
print("Local Jacobian condition/rank — previous versus repaired signals")
display(signal_identifiability.round(6))
print("Parameters at bounds (active_bound) — previous versus repaired signals")
display(
    signal_parameter_changes[
        [
            "calibration_matrix",
            "parameter",
            "active_bound_A_previous_signals",
            "active_bound_B_repaired_signals",
        ]
    ]
)
print("Activation-parameter LOO stability — previous versus repaired signals")
display(signal_loo_summary.round(6))


## LAB013–LAB015 — external kinetic / activation holdout

This block replaces the former LAB016–LAB018 gas-phase holdout. It does **not** modify the
`no_lab010` calibration above: natural theta remains the frozen complete 17-parameter vector,
the CO2 layer remains the current fitted natural parameter set, LAB010 remains excluded, and
LAB012 remains the independent gas-phase CO2 holdout.

LAB013–LAB015 gas-phase CO2 is documented as unreliable. It is therefore excluded from
calibration, scores, integrals, peak metrics and parameter estimation. At the user's request,
the filtered sensor curves are loaded only for a clearly labelled exploratory visual overlay
against the frozen prediction. LAB013–LAB015 contribute no residual and no parameter is
re-estimated in this section.

### Time-zero policy

Repository logs contain no defensible inoculation timestamp for these three reactors. The
existing data convention sets `t_proxy = 0` at numbered reactor sample 1 (2026-08-18 10:00),
which is a synchronization proxy—not inoculation. Temperature logging begins near 11:20,
gas logging near 11:22, and `nutricion_activa` has no edge. A prior repository audit also marks
the inoculation hour as undocumented. Therefore:

- state/shape comparisons on the common sample clock are valid;
- bracket widths and ordering are valid;
- absolute lag, activation time from inoculation and gas onset from inoculation are **not** valid;
- all reported times below carry the suffix `_proxy_h` and must not be read as biological age.


In [ ]:
from shared import run_new_must_glycerol_estimability_doe as kinetic_base

EXTERNAL_KINETIC_LABS = ("LAB013", "LAB014", "LAB015")
LAB_TO_FERMENTER = {"LAB013": "F1", "LAB014": "F2", "LAB015": "F3"}
EXTERNAL_KINETIC_DIR = FM / "data" / "mem2026" / "LAB013-015"
OFFLINE_PATH = EXTERNAL_KINETIC_DIR / "LAB013-LAB015-offline-measurements.csv"
COMBINED_PATH = EXTERNAL_KINETIC_DIR / "LAB013-LAB015-offline-combined.csv"
Y15_PATH = EXTERNAL_KINETIC_DIR / "Y15_LAB013-015.csv"
PRIOR_T0_AUDIT_PATH = analysis.NOTEBOOK_DIR / "natural_must_initial_conditions_analysis.ipynb"
GAS_PATHS_EXCLUDED = {
    "LAB013": EXTERNAL_KINETIC_DIR / "CO2_FILT_F1_LAB013.csv",
    "LAB014": EXTERNAL_KINETIC_DIR / "CO2_FILT_F2_LAB014.csv",
    "LAB015": EXTERNAL_KINETIC_DIR / "CO2_FILT_F3_LAB015.csv",
}
TEMP_PATHS = {
    lab: EXTERNAL_KINETIC_DIR / f"Temp_{LAB_TO_FERMENTER[lab]}_{lab}.csv"
    for lab in EXTERNAL_KINETIC_LABS
}
OCULYZE_PATHS = {
    lab: EXTERNAL_KINETIC_DIR / f"{lab}_OCULYZE" / "report.csv"
    for lab in EXTERNAL_KINETIC_LABS
}

source_rows = [
    {"source": "offline ledger / Brix / density / DO / dissolved CO2", "path": OFFLINE_PATH},
    {"source": "offline combined ledger + Y15", "path": COMBINED_PATH},
    {"source": "Y15 PI and temporal chemistry", "path": Y15_PATH},
    {"source": "prior t0 audit", "path": PRIOR_T0_AUDIT_PATH},
]
for lab in EXTERNAL_KINETIC_LABS:
    source_rows.extend([
        {"source": f"{lab} measured temperature", "path": TEMP_PATHS[lab]},
        {"source": f"{lab} Oculyze X/Xd", "path": OCULYZE_PATHS[lab]},
        {"source": f"{lab} gas CO2 — EXCLUDED", "path": GAS_PATHS_EXCLUDED[lab]},
    ])
external_source_audit = pd.DataFrame(source_rows)
external_source_audit["exists"] = external_source_audit["path"].map(Path.exists)
external_source_audit["used_for_state_holdout"] = ~external_source_audit["source"].str.contains("EXCLUDED")
external_source_audit["used_for_exploratory_gas_plot"] = external_source_audit["source"].str.contains("gas CO2")
external_source_audit["relative_path"] = external_source_audit["path"].map(
    lambda path: path.relative_to(ROOT).as_posix()
)
assert external_source_audit["exists"].all()
assert not external_source_audit.loc[
    external_source_audit["source"].str.contains("gas CO2"), "used_for_state_holdout"
].any()
display(external_source_audit.drop(columns="path"))

offline_13_15 = pd.read_csv(OFFLINE_PATH)
combined_13_15 = pd.read_csv(COMBINED_PATH)
y15_13_15 = pd.read_csv(Y15_PATH)
for frame in (offline_13_15, combined_13_15):
    frame["sample_datetime"] = pd.to_datetime(
        frame["fecha_muestreo"].astype(str) + " " + frame["hora_muestreo"].astype(str),
        errors="raise",
    )
y15_13_15["y15_datetime"] = pd.to_datetime(y15_13_15["y15_datetime"], errors="raise")
y15_13_15["y15_datetime_end"] = pd.to_datetime(y15_13_15["y15_datetime_end"], errors="coerce")

proxy_t0_by_lab = (
    offline_13_15[offline_13_15["sample_id"].str.endswith("-1")]
    .set_index("lab_id")["sample_datetime"].to_dict()
)
assert set(proxy_t0_by_lab) == set(EXTERNAL_KINETIC_LABS)
for frame in (offline_13_15, combined_13_15):
    frame["time_proxy_h"] = [
        (stamp - proxy_t0_by_lab[lab]).total_seconds() / 3600.0
        for lab, stamp in zip(frame["lab_id"], frame["sample_datetime"])
    ]

pi_13_15 = y15_13_15[y15_13_15["sample_type"].eq("pre_inoculum")].copy()
assert set(pi_13_15["lab_id"]) == set(EXTERNAL_KINETIC_LABS)

temperature_native = {}
t0_audit_rows = []
for lab in EXTERNAL_KINETIC_LABS:
    temp = pd.read_csv(TEMP_PATHS[lab])
    temp["timestamp"] = pd.to_datetime(temp["timestamp"], errors="raise")
    temp["T"] = pd.to_numeric(temp["T"], errors="raise")
    active = pd.to_numeric(temp["nutricion_activa"], errors="coerce").fillna(0)
    assert active.eq(0).all(), f"{lab}: an undeclared nutrition edge exists"
    temp["time_proxy_h"] = (
        temp["timestamp"] - proxy_t0_by_lab[lab]
    ).dt.total_seconds() / 3600.0
    temperature_native[lab] = temp
    pi_row = pi_13_15[pi_13_15["lab_id"].eq(lab)].iloc[0]
    first_dynamic = y15_13_15[
        y15_13_15["lab_id"].eq(lab) & y15_13_15["sample_type"].eq("fermentation_sample")
    ]["y15_datetime"].min()
    first_oculyze = pd.read_csv(OCULYZE_PATHS[lab]).iloc[0]
    t0_audit_rows.append({
        "batch": lab,
        "proxy_origin": proxy_t0_by_lab[lab],
        "proxy_definition": "numbered reactor sample 1; NOT inoculation",
        "first_temperature_log": temp["timestamp"].min(),
        "first_oculyze_process_description": str(first_oculyze["Description"]),
        "pi_y15_start": pi_row["y15_datetime"],
        "pi_y15_end": pi_row["y15_datetime_end"],
        "first_temporal_y15_analysis": first_dynamic,
        "nutrition_edges": 0,
        "inoculation_t0_resolved": False,
        "absolute_onset_valid": False,
    })
t0_audit = pd.DataFrame(t0_audit_rows)
display(t0_audit)
print("T0 VERDICT: inoculation time is unresolved; sample-1 proxy clock retained only for synchronized state/shape diagnostics.")


### Frozen kinetic holdout construction

The initial composition uses each pre-inoculum Y15 row. Viable and dead biomass use the first
Oculyze sample with the repository conversion
`1 million cells/mL = 0.03 g/L`; subsequent Oculyze points are validation observations.
The duplicated `LAB015-1` identifier is aligned to the nearest unused ledger timestamp, without
editing the source. Measured temperature is interpolated on the 0.25 h driver grid. No nutrient
pulse is introduced. Ethanol is unavailable and remains unobserved (`NaN`).

This is a conditional external holdout: parameters are frozen, but measured initial conditions
are legitimate experiment inputs. The gas-phase CO2 files remain outside kinetic construction, fitting and scoring; they are read
later only for the explicitly exploratory observed-versus-predicted overlay.


In [ ]:
def _align_oculyze(lab, path, schedule):
    report = pd.read_csv(path).copy()
    available = schedule[schedule["lab_id"].eq(lab)].copy()
    available_by_id = available.set_index("sample_id")["sample_datetime"].to_dict()
    used, aligned_ids, notes = set(), [], []
    for item in report.itertuples(index=False):
        raw_id = str(item.Name).strip()
        if raw_id in available_by_id and raw_id not in used:
            chosen, note = raw_id, "matched unique sample ID to offline ledger"
        else:
            parsed = pd.to_datetime(str(item.Description), dayfirst=True, errors="coerce")
            candidates = available[~available["sample_id"].isin(used)].copy()
            if pd.isna(parsed) or candidates.empty:
                raise ValueError(f"{lab}: cannot align Oculyze row {raw_id}")
            distance = (candidates["sample_datetime"] - parsed).abs()
            chosen = str(candidates.loc[distance.idxmin(), "sample_id"])
            note = f"repaired duplicate label {raw_id} by nearest unused ledger timestamp"
        used.add(chosen)
        aligned_ids.append(chosen)
        notes.append(note)
    report["batch"] = lab
    report["sample_id_raw"] = report["Name"].astype(str)
    report["sample_id"] = aligned_ids
    report["timestamp_source"] = notes
    report = report.merge(
        available[["sample_id", "sample_datetime"]], on="sample_id", how="left", validate="one_to_one"
    )
    report["Concentration"] = pd.to_numeric(report["Concentration"], errors="raise")
    report["Viability"] = pd.to_numeric(report["Viability"], errors="raise")
    report["X_total_g_l"] = report["Concentration"] * 0.03
    report["X_obs_g_l"] = report["X_total_g_l"] * report["Viability"] / 100.0
    report["Xd_obs_g_l"] = report["X_total_g_l"] - report["X_obs_g_l"]
    report["time_proxy_h"] = (
        report["sample_datetime"] - proxy_t0_by_lab[lab]
    ).dt.total_seconds() / 3600.0
    return report

oculyze_13_15 = pd.concat(
    [_align_oculyze(lab, OCULYZE_PATHS[lab], offline_13_15) for lab in EXTERNAL_KINETIC_LABS],
    ignore_index=True,
)

theta_sets_external, theta_validation_external, _ = analysis.load_complete_theta_sets()
theta_natural_frozen = theta_sets_external["natural"]
assert len(theta_natural_frozen) == 17
fixed_co2_parameters = dict(result["fits"]["natural"])
assert fixed_co2_parameters["model"] == analysis.MODEL_NAME

for column in ["YAN_mg_L", "glucose_g_L", "fructose_g_L", "glycerol_g_L"]:
    y15_13_15[column] = pd.to_numeric(y15_13_15[column], errors="coerce")
for column in ["YAN_mg_L", "glucose_g_L", "fructose_g_L", "glycerol_g_L"]:
    combined_13_15[column] = pd.to_numeric(combined_13_15[column], errors="coerce")

external_kinetic_batches = {}
external_temperature_inputs = []
initial_condition_rows = []
for lab in EXTERNAL_KINETIC_LABS:
    chemistry = combined_13_15[combined_13_15["lab_id"].eq(lab)].dropna(
        subset=["glucose_g_L", "fructose_g_L"]
    ).sort_values("time_proxy_h")
    biomass = oculyze_13_15[oculyze_13_15["batch"].eq(lab)].sort_values("time_proxy_h")
    horizon_h = float(max(chemistry["time_proxy_h"].max(), biomass["time_proxy_h"].max()))
    grid = np.arange(0.0, horizon_h + _historical.DRIVER_GRID_H, _historical.DRIVER_GRID_H)
    model_time = np.unique(np.concatenate([
        grid[grid <= horizon_h + 1e-9],
        chemistry["time_proxy_h"].to_numpy(float),
        biomass["time_proxy_h"].to_numpy(float),
        np.array([horizon_h]),
    ]))
    temp = temperature_native[lab].dropna(subset=["timestamp", "T"]).copy()
    temp = (
        temp.set_index("timestamp")["T"].resample("30min").median().dropna().reset_index()
    )
    temp["time_proxy_h"] = (
        temp["timestamp"] - proxy_t0_by_lab[lab]
    ).dt.total_seconds() / 3600.0
    first_offline_temp = float(pd.to_numeric(
        offline_13_15.loc[
            offline_13_15["sample_id"].eq(f"{lab}-1"), "temperature_C"
        ], errors="raise"
    ).iloc[0])
    temp_grid = pd.concat([
        pd.DataFrame({"timestamp": [proxy_t0_by_lab[lab]], "T": [first_offline_temp], "time_proxy_h": [0.0]}),
        temp[["timestamp", "T", "time_proxy_h"]],
    ], ignore_index=True).sort_values("time_proxy_h").drop_duplicates("time_proxy_h")
    temperature = np.interp(model_time, temp_grid["time_proxy_h"], temp_grid["T"])
    external_temperature_inputs.append(pd.DataFrame({
        "batch": lab, "time_proxy_h": model_time, "temperature_c": temperature
    }))

    observations = {state: np.full(len(model_time), np.nan) for state in kinetic_base.STATE_NAMES}
    for row in chemistry.itertuples(index=False):
        idx = int(np.argmin(np.abs(model_time - float(row.time_proxy_h))))
        observations["G"][idx] = float(row.glucose_g_L)
        observations["F"][idx] = float(row.fructose_g_L)
        observations["N"][idx] = float(row.YAN_mg_L) * 1e-3
        observations["Gly"][idx] = float(row.glycerol_g_L)
    for row in biomass.itertuples(index=False):
        idx = int(np.argmin(np.abs(model_time - float(row.time_proxy_h))))
        observations["X"][idx] = float(row.X_obs_g_l)
        observations["Xd"][idx] = float(row.Xd_obs_g_l)

    pi_row = pi_13_15[pi_13_15["lab_id"].eq(lab)].iloc[0]
    first_biomass = biomass.iloc[0]
    initials = {
        "X": float(first_biomass["X_obs_g_l"]),
        "Xd": float(first_biomass["Xd_obs_g_l"]),
        "N": float(pi_row["YAN_mg_L"]) * 1e-3,
        "G": float(pi_row["glucose_g_L"]),
        "F": float(pi_row["fructose_g_L"]),
        "E": 0.0,
        "Gly": float(pi_row["glycerol_g_L"]),
    }
    external_kinetic_batches[lab] = kinetic_base.BatchData(
        medium="natural", batch=lab, time=model_time, temperature_c=temperature,
        pulses={channel: tuple() for channel in kinetic_base.INPUT_CHANNELS},
        observations=observations, initials=initials,
    )
    initial_condition_rows.append({"batch": lab, **initials})

external_initial_conditions = pd.DataFrame(initial_condition_rows)
external_temperature_inputs = pd.concat(external_temperature_inputs, ignore_index=True)
external_state_coverage = pd.DataFrame([
    {
        "batch": lab,
        "X": int(oculyze_13_15["batch"].eq(lab).sum()),
        "Xd": int(oculyze_13_15["batch"].eq(lab).sum()),
        "N_YAN": int((combined_13_15["lab_id"].eq(lab) & combined_13_15["YAN_mg_L"].notna()).sum()),
        "G": int((combined_13_15["lab_id"].eq(lab) & combined_13_15["glucose_g_L"].notna()).sum()),
        "F": int((combined_13_15["lab_id"].eq(lab) & combined_13_15["fructose_g_L"].notna()).sum()),
        "Gly": int((combined_13_15["lab_id"].eq(lab) & combined_13_15["glycerol_g_L"].notna()).sum()),
        "E": 0,
        "gas_CO2_used": False,
    }
    for lab in EXTERNAL_KINETIC_LABS
])
display(external_initial_conditions.round(5))
display(external_state_coverage)


### State predictions and holdout metrics

The first Oculyze point supplies `X0/Xd0` and is excluded from X/Xd scores because it is an
initialization anchor. Pre-inoculum Y15 supplies `N0/G0/F0/Gly0`; only numbered temporal samples
are scored. NRMSE is RMSE divided by the observed range. `R²` is descriptive and can be strongly
negative when the frozen trajectory is worse than the observed mean.


In [ ]:
external_state_predictions = {
    lab: kinetic_base.simulate(batch, theta_natural_frozen, batch.time)
    for lab, batch in external_kinetic_batches.items()
}
assert all(frame is not None for frame in external_state_predictions.values())

point_rows = []
state_specs = {
    "G": ("glucose_g_L", 1.0, "g/L"),
    "F": ("fructose_g_L", 1.0, "g/L"),
    "N": ("YAN_mg_L", 1000.0, "mg/L"),
    "Gly": ("glycerol_g_L", 1.0, "g/L"),
}
for lab in EXTERNAL_KINETIC_LABS:
    sim = external_state_predictions[lab]
    chem = combined_13_15[combined_13_15["lab_id"].eq(lab)].sort_values("time_proxy_h")
    for state, (column, factor, unit) in state_specs.items():
        observed_rows = chem.dropna(subset=[column])
        for row in observed_rows.itertuples(index=False):
            predicted = factor * float(np.interp(row.time_proxy_h, sim.index, sim[state]))
            observed = float(getattr(row, column))
            point_rows.append({
                "batch": lab, "state": state, "unit": unit,
                "sample_id": row.sample_id, "time_proxy_h": float(row.time_proxy_h),
                "observed": observed, "predicted": predicted,
                "error_pred_minus_obs": predicted - observed,
                "initialization_anchor": False,
            })
    biomass = oculyze_13_15[oculyze_13_15["batch"].eq(lab)].sort_values("time_proxy_h")
    for state, column in (("X", "X_obs_g_l"), ("Xd", "Xd_obs_g_l")):
        for row in biomass.itertuples(index=False):
            predicted = float(np.interp(row.time_proxy_h, sim.index, sim[state]))
            observed = float(getattr(row, column))
            point_rows.append({
                "batch": lab, "state": state, "unit": "g/L",
                "sample_id": row.sample_id, "time_proxy_h": float(row.time_proxy_h),
                "observed": observed, "predicted": predicted,
                "error_pred_minus_obs": predicted - observed,
                "initialization_anchor": bool(abs(row.time_proxy_h) < 1e-9),
            })

external_state_point_errors = pd.DataFrame(point_rows)
metric_rows = []
for (lab, state), group in external_state_point_errors.groupby(["batch", "state"], sort=True):
    scored = group[~group["initialization_anchor"]].sort_values("time_proxy_h")
    observed = scored["observed"].to_numpy(float)
    predicted = scored["predicted"].to_numpy(float)
    error = predicted - observed
    observed_range = float(np.ptp(observed)) if len(observed) else np.nan
    ss_tot = float(np.sum((observed - observed.mean()) ** 2)) if len(observed) else np.nan
    metric_rows.append({
        "batch": lab, "state": state, "unit": scored["unit"].iloc[0], "n": len(scored),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "nrmse_observed_range": float(np.sqrt(np.mean(error ** 2))) / observed_range if observed_range > 0 else np.nan,
        "bias_pred_minus_obs": float(np.mean(error)),
        "r2": 1.0 - float(np.sum(error ** 2)) / ss_tot if ss_tot > 0 else np.nan,
        "first_time_proxy_h": float(scored["time_proxy_h"].iloc[0]),
        "observed_initial": float(observed[0]), "predicted_initial": float(predicted[0]),
        "final_time_proxy_h": float(scored["time_proxy_h"].iloc[-1]),
        "observed_final": float(observed[-1]), "predicted_final": float(predicted[-1]),
    })
external_kinetic_metrics = pd.DataFrame(metric_rows)
display(external_kinetic_metrics.round(5))
display(external_state_point_errors.round(5))

state_labels = {
    "X": "viable X [g/L]", "Xd": "dead Xd [g/L]", "N": "YAN / N [mg/L]",
    "G": "glucose G [g/L]", "F": "fructose F [g/L]", "Gly": "glycerol [g/L]",
}
for lab in EXTERNAL_KINETIC_LABS:
    sim = external_state_predictions[lab]
    fig, axes = plt.subplots(3, 2, figsize=(12.5, 10.5), sharex=True)
    for ax, state in zip(axes.flat, ("X", "Xd", "N", "G", "F", "Gly")):
        factor = 1000.0 if state == "N" else 1.0
        points = external_state_point_errors[
            external_state_point_errors["batch"].eq(lab) & external_state_point_errors["state"].eq(state)
        ]
        ax.plot(sim.index, factor * sim[state], color="tab:blue", lw=2.0, label="frozen full-theta prediction")
        ax.scatter(points["time_proxy_h"], points["observed"], color="black", s=30, zorder=3, label="observation")
        anchors = points[points["initialization_anchor"]]
        if not anchors.empty:
            ax.scatter(anchors["time_proxy_h"], anchors["observed"], marker="s", facecolors="none",
                       edgecolors="tab:orange", s=65, zorder=4, label="initial-condition anchor")
        ax.set_ylabel(state_labels[state]); ax.grid(alpha=0.22)
    axes[-1, 0].set_xlabel("time from numbered sample 1 [proxy h]")
    axes[-1, 1].set_xlabel("time from numbered sample 1 [proxy h]")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.suptitle(
        f"{lab} — external kinetic holdout (gas CO2 excluded from scoring)",
        y=0.992, fontsize=14,
    )
    fig.legend(
        handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.958),
        ncol=3, frameon=False, fontsize=9,
    )
    fig.tight_layout(rect=(0.025, 0.02, 0.975, 0.90))
    fig.savefig(analysis.PLOT_DIR / f"external_kinetic_holdout_{lab}_states.png", dpi=180, bbox_inches="tight")
    plt.show()


### Chemical bracket, activation gate and CO2-source chain

The bracket uses the unchanged historical rule on paired temporal G/F samples: the first point
where `G + F` has fallen by at least 5 g/L from the first temporal G/F observation, with the
previous temporal sample as the lower bound. Temporal ethanol is absent, so the ethanol rule is
not applied. Frozen CO2 parameters then place the smooth gate inside that measured bracket.

`qCO2_base → gate/O2 → qCO2_effective → dissolved pool → release` is shown without gas-phase
ground truth. Predicted release is a model diagnostic only.


In [ ]:
external_cache_support = pd.DataFrame([
    {
        "batch": lab, "matrix": "natural", "calibratable": True,
        "chemistry_last_h": float(batch.time[-1]), "t_h": 0.0,
    }
    for lab, batch in external_kinetic_batches.items()
])
external_driver_cache, external_driver_diagnostics = _historical.build_driver_cache(
    external_kinetic_batches, {"natural": theta_natural_frozen}, external_cache_support
)

activation_rows, chain_frames, termination_rows = [], [], []
for lab, cache in external_driver_cache.items():
    parameters = fixed_co2_parameters
    qgas_raw, qprod_effective, o2_internal, phi_ana = _historical.raw_qgas_grid_prediction(
        cache,
        parameters["kCO2_release_h"], parameters["CO2sat_scale"],
        parameters["O2_qmax_mg_gdw_h"], parameters["O2_initial_scale"],
        chemistry_aligned=True, nitrogen_boost_transition=True,
        pulse_t_rise_h=parameters["pulse_t_rise_h"],
        pulse_activity_gain=parameters["pulse_activity_gain"],
        continuous_release=True, bounded_chemical_activation=True,
        chem_activation_start_fraction=parameters["chem_activation_start_fraction"],
        chem_activation_duration_fraction=parameters["chem_activation_duration_fraction"],
    )
    driver = external_driver_diagnostics[external_driver_diagnostics["batch"].eq(lab)].iloc[0]
    lower_h, upper_h = float(driver.chemical_activity_lower_h), float(driver.chemical_activity_upper_h)
    width_h = upper_h - lower_h
    gate_start_h = lower_h + parameters["chem_activation_start_fraction"] * width_h
    gate_duration_h = max(parameters["chem_activation_duration_fraction"] * width_h, _historical.DRIVER_GRID_H)
    gate_end_h = gate_start_h + gate_duration_h
    gate = _historical._bounded_smoothstep_activation(
        cache.time_h, lower_h, upper_h,
        parameters["chem_activation_start_fraction"],
        parameters["chem_activation_duration_fraction"],
    )
    dissolved = np.zeros(len(cache.time_h), dtype=float)
    for idx in range(1, len(cache.time_h)):
        dt = float(cache.time_h[idx] - cache.time_h[idx - 1])
        dissolved[idx] = max(dissolved[idx - 1] + dt * (qprod_effective[idx - 1] - qgas_raw[idx]), 0.0)
    qgas_scaled = parameters["matrix_gain"] * qgas_raw

    chemistry = combined_13_15[
        combined_13_15["lab_id"].eq(lab) & combined_13_15["glucose_g_L"].notna()
        & combined_13_15["fructose_g_L"].notna()
    ].sort_values("time_proxy_h")
    sugar = chemistry["glucose_g_L"].to_numpy(float) + chemistry["fructose_g_L"].to_numpy(float)
    crossed = sugar <= sugar[0] - _historical.CHEMISTRY_SUGAR_DROP_G_L
    first_cross = int(np.flatnonzero(crossed)[0])
    assert np.isclose(lower_h, chemistry["time_proxy_h"].iloc[first_cross - 1])
    assert np.isclose(upper_h, chemistry["time_proxy_h"].iloc[first_cross])

    first_qprod_base = _historical._sustained_onset_h(
        cache.time_h, cache.base_qprod_g_l_h,
        _historical.EARLY_EMISSION_THRESHOLD_G_L_H, consecutive=2
    )
    first_qprod_effective = _historical._sustained_onset_h(
        cache.time_h, qprod_effective,
        _historical.EARLY_EMISSION_THRESHOLD_G_L_H, consecutive=2
    )
    first_qgas = _historical._sustained_onset_h(
        cache.time_h, qgas_scaled, _historical.EARLY_EMISSION_THRESHOLD_G_L_H, consecutive=2
    )
    gas_detectable = _historical._sustained_onset_h(
        cache.time_h, qgas_scaled, _historical.CO2_DETECTION_LIMIT_G_L_H, consecutive=3
    )
    activation_rows.append({
        "batch": lab,
        "chemical_activity_lower_proxy_h": lower_h,
        "chemical_activity_upper_proxy_h": upper_h,
        "bracket_width_h": width_h,
        "criterion": "G+F drop >= 5 g/L; temporal E unavailable",
        "gate_start_proxy_h": gate_start_h,
        "gate_end_proxy_h": gate_end_h,
        "gate_duration_h": gate_duration_h,
        "qCO2_base_peak_g_l_h": float(np.max(cache.base_qprod_g_l_h)),
        "qCO2_effective_peak_g_l_h": float(np.max(qprod_effective)),
        "first_base_metabolic_qCO2_gt_0p005_proxy_h": first_qprod_base,
        "first_effective_qCO2_after_gate_gt_0p005_proxy_h": first_qprod_effective,
        "first_gas_release_gt_0p005_proxy_h": first_qgas,
        "predicted_gas_onset_over_operational_LOD_proxy_h": gas_detectable,
        "absolute_time_from_inoculation_valid": False,
    })
    sim = external_state_predictions[lab]
    chain = pd.DataFrame({
        "batch": lab, "time_proxy_h": cache.time_h,
        "qCO2_base_g_l_h": cache.base_qprod_g_l_h,
        "chemical_gate": gate,
        "o2_internal_mg_l": o2_internal,
        "anaerobic_fraction": phi_ana,
        "qCO2_effective_g_l_h": qprod_effective,
        "dissolved_CO2_pool_g_l": dissolved,
        "qgas_release_raw_g_l_h": qgas_raw,
        "qgas_release_scaled_g_l_h": qgas_scaled,
    })
    for state in kinetic_base.STATE_NAMES:
        chain[state] = np.interp(cache.time_h, sim.index, sim[state])
    chain_frames.append(chain)

    peak_idx = int(np.argmax(qprod_effective))
    post_peak = np.arange(len(qprod_effective)) >= peak_idx
    below_tail = post_peak & (qprod_effective <= 0.10 * max(float(np.max(qprod_effective)), 1e-12))
    tail_time = float(cache.time_h[np.flatnonzero(below_tail)[0]]) if below_tail.any() else np.nan
    last_12 = cache.time_h >= cache.time_h[-1] - 12.0
    points = external_state_point_errors[external_state_point_errors["batch"].eq(lab)]
    def _final_pair(state):
        row = points[points["state"].eq(state)].sort_values("time_proxy_h").iloc[-1]
        return float(row.observed), float(row.predicted), float(row.time_proxy_h)
    g_obs, g_pred, g_t = _final_pair("G")
    f_obs, f_pred, f_t = _final_pair("F")
    n_obs, n_pred, n_t = _final_pair("N")
    gly_obs, gly_pred, gly_t = _final_pair("Gly")
    termination_rows.append({
        "batch": lab, "last_chemistry_proxy_h": max(g_t, f_t, n_t, gly_t),
        "observed_final_GplusF_g_l": g_obs + f_obs,
        "predicted_final_GplusF_g_l": g_pred + f_pred,
        "observed_final_YAN_mg_l": n_obs, "predicted_final_N_mg_l": n_pred,
        "observed_final_Gly_g_l": gly_obs, "predicted_final_Gly_g_l": gly_pred,
        "qCO2_effective_peak_proxy_h": float(cache.time_h[peak_idx]),
        "qCO2_below_10pct_peak_proxy_h": tail_time,
        "mean_last12h_qCO2_over_peak": float(np.mean(qprod_effective[last_12]) / max(np.max(qprod_effective), 1e-12)),
        "final_qgas_over_peak": float(qgas_scaled[-1] / max(np.max(qgas_scaled), 1e-12)),
        "absolute_termination_time_valid": False,
    })

external_activation_summary = pd.DataFrame(activation_rows)
external_chain_trajectories = pd.concat(chain_frames, ignore_index=True)
external_termination_summary = pd.DataFrame(termination_rows)
display(external_activation_summary.round(5))
display(external_termination_summary.round(5))

for lab in EXTERNAL_KINETIC_LABS:
    chain = external_chain_trajectories[external_chain_trajectories["batch"].eq(lab)]
    summary = external_activation_summary.set_index("batch").loc[lab]
    fig, axes = plt.subplots(5, 1, figsize=(12.5, 13), sharex=True)
    axes[0].plot(chain["time_proxy_h"], chain["qCO2_base_g_l_h"], color="0.45", lw=1.6, label="qCO2_base")
    axes[0].plot(chain["time_proxy_h"], chain["qCO2_effective_g_l_h"], color="tab:blue", lw=2.0, label="after gate + O2")
    axes[1].plot(chain["time_proxy_h"], chain["chemical_gate"], color="tab:orange", lw=2.0, label="chemical gate")
    axes[2].plot(chain["time_proxy_h"], chain["o2_internal_mg_l"], color="tab:green", lw=2.0, label="internal O2")
    axes[2].plot(chain["time_proxy_h"], chain["anaerobic_fraction"], color="tab:red", lw=1.5, ls="--", label="anaerobic fraction")
    axes[3].plot(chain["time_proxy_h"], chain["dissolved_CO2_pool_g_l"], color="tab:purple", lw=2.0, label="latent dissolved pool")
    axes[4].plot(chain["time_proxy_h"], chain["qgas_release_scaled_g_l_h"], color="tab:blue", lw=2.0, label="predicted release — gas overlay shown separately")
    for ax in axes:
        ax.axvspan(summary.chemical_activity_lower_proxy_h, summary.chemical_activity_upper_proxy_h,
                   color="tab:orange", alpha=0.10, label="chemical bracket")
        ax.grid(alpha=0.22); ax.legend(frameon=False, fontsize=8, loc="upper right")
    axes[0].set_ylabel("CO2 [g/L/h]"); axes[1].set_ylabel("gate [-]")
    axes[2].set_ylabel("O2 [mg/L] / fraction")
    axes[3].set_ylabel("pool [g/L]"); axes[4].set_ylabel("qgas [g/L/h]")
    axes[-1].set_xlabel("time from numbered sample 1 [proxy h]")
    fig.suptitle(f"{lab} — frozen state → CO2-source → release chain", y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(analysis.PLOT_DIR / f"external_activation_chain_{lab}.png", dpi=180, bbox_inches="tight")
    plt.show()


### Exploratory gas-phase CO2 overlay — visualization only

This is the requested visual comparison, not a validation score. The documented-unreliable
`CO2_FILT_*` signal is aligned to the same sample-1 proxy clock, corrected with the current
run-specific initial-12-h q10 zero rule, converted with `SCCM_CORRECTED` at the documented 2 L
reactor volume, aggregated hourly and passed through the current median + Savitzky–Golay
smoothing. No additional transient repair is inferred for these LABs.

The observed curve is not passed to the optimizer and is not used for RMSE, integral, peak,
onset or parameter conclusions. The orange band is the G+F chemical bracket; it is provided
only as temporal context. Because inoculation t0 remains unresolved, the horizontal coordinate
is still time from numbered sample 1 rather than biological age.


In [ ]:
external_gas_frames = []
external_gas_audit_rows = []
for lab in EXTERNAL_KINETIC_LABS:
    raw_gas = pd.read_csv(
        GAS_PATHS_EXCLUDED[lab],
        usecols=["timestamp", "flow_filt_sccm", "status", "is_spike"],
    )
    raw_gas["timestamp"] = pd.to_datetime(raw_gas["timestamp"], errors="raise")
    raw_gas["flow_filt_sccm"] = pd.to_numeric(raw_gas["flow_filt_sccm"], errors="coerce")
    raw_gas["is_spike"] = pd.to_numeric(raw_gas["is_spike"], errors="coerce").fillna(0)
    raw_gas = raw_gas.dropna(subset=["timestamp", "flow_filt_sccm"]).sort_values("timestamp")
    raw_gas["batch"] = lab
    raw_gas["time_proxy_h"] = (
        raw_gas["timestamp"] - proxy_t0_by_lab[lab]
    ).dt.total_seconds() / 3600.0
    prediction_horizon = float(
        external_chain_trajectories.loc[
            external_chain_trajectories["batch"].eq(lab), "time_proxy_h"
        ].max()
    )
    raw_gas = raw_gas[raw_gas["time_proxy_h"].between(0.0, prediction_horizon)].copy()
    first_available_h = float(raw_gas["time_proxy_h"].min())
    zero_candidates = raw_gas.loc[
        raw_gas["time_proxy_h"].le(first_available_h + _historical.SENSOR_ZERO_WINDOW_H),
        "flow_filt_sccm",
    ]
    zero_offset_sccm = max(
        0.0, float(zero_candidates.quantile(_historical.SENSOR_ZERO_QUANTILE))
    )
    raw_gas["zero_corrected_sccm"] = (
        raw_gas["flow_filt_sccm"] - zero_offset_sccm
    ).clip(lower=0.0)
    raw_gas["co2_rate_zero_corrected_g_l_h"] = (
        raw_gas["zero_corrected_sccm"] * analysis.SCCM_FACTOR_G_L_H_PER_SCCM
    )
    raw_gas["time_bin_h"] = (
        raw_gas["time_proxy_h"] / _historical.OBSERVATION_BIN_H
    ).round() * _historical.OBSERVATION_BIN_H
    hourly_gas = (
        raw_gas.groupby("time_bin_h", as_index=False)
        .agg(
            time_proxy_h=("time_proxy_h", "median"),
            flow_filt_sccm=("flow_filt_sccm", "median"),
            zero_corrected_sccm=("zero_corrected_sccm", "median"),
            co2_rate_filtered_g_l_h=("co2_rate_zero_corrected_g_l_h", "median"),
            spike_fraction=("is_spike", "mean"),
            n_raw=("timestamp", "size"),
        )
        .sort_values("time_proxy_h")
    )
    hourly_gas["matrix"] = "natural"
    hourly_gas["batch"] = lab
    hourly_gas["t_h"] = hourly_gas["time_proxy_h"]
    hourly_gas = _historical.smooth_hourly_co2_profiles(
        hourly_gas,
        pd.DataFrame(columns=["batch", "pulse_time_h"]),
    )
    hourly_gas["co2_observed_exploratory_g_l_h"] = hourly_gas[
        "co2_rate_smoothed_g_l_h"
    ]
    hourly_gas["sensor_zero_offset_sccm"] = zero_offset_sccm
    hourly_gas["sccm_conversion"] = "SCCM_CORRECTED"
    hourly_gas["used_in_fit"] = False
    hourly_gas["used_in_metrics"] = False
    external_gas_frames.append(hourly_gas)
    external_gas_audit_rows.append({
        "batch": lab,
        "source_file": GAS_PATHS_EXCLUDED[lab].relative_to(ROOT).as_posix(),
        "n_raw": len(raw_gas),
        "n_hourly": len(hourly_gas),
        "first_time_proxy_h": first_available_h,
        "last_time_proxy_h": float(raw_gas["time_proxy_h"].max()),
        "sensor_zero_offset_sccm": zero_offset_sccm,
        "sccm_factor_g_l_h_per_sccm": analysis.SCCM_FACTOR_G_L_H_PER_SCCM,
        "used_in_fit": False,
        "used_in_metrics": False,
        "reliability": "documented unreliable; visualization only",
    })

external_gas_observed = pd.concat(external_gas_frames, ignore_index=True)
external_gas_plot_audit = pd.DataFrame(external_gas_audit_rows)
assert not external_gas_observed["used_in_fit"].any()
assert not external_gas_observed["used_in_metrics"].any()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), sharex=True, sharey=True)
for ax, lab in zip(axes, EXTERNAL_KINETIC_LABS):
    observed = external_gas_observed[external_gas_observed["batch"].eq(lab)]
    predicted = external_chain_trajectories[
        external_chain_trajectories["batch"].eq(lab)
    ]
    bracket = external_activation_summary.set_index("batch").loc[lab]
    ax.plot(
        observed["time_proxy_h"], observed["co2_observed_exploratory_g_l_h"],
        color="black", lw=1.8, label="observed CO2 (unreliable; visual only)",
    )
    ax.plot(
        predicted["time_proxy_h"], predicted["qgas_release_scaled_g_l_h"],
        color="tab:blue", lw=2.2, label="frozen predicted qgas",
    )
    ax.axvspan(
        bracket.chemical_activity_lower_proxy_h,
        bracket.chemical_activity_upper_proxy_h,
        color="tab:orange", alpha=0.13, label="G+F chemical bracket",
    )
    ax.set_title(lab, fontsize=13, pad=10)
    ax.set_xlabel("time from numbered sample 1 [proxy h]")
    ax.grid(alpha=0.22)
    ax.text(
        0.02, 0.97, "NO FIT · NO SCORE",
        transform=ax.transAxes, ha="left", va="top",
        color="crimson", fontsize=9, fontweight="bold",
    )
axes[0].set_ylabel("CO2 rate [g/L/h]")
handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle(
    "LAB013–LAB015 — observed versus frozen predicted gas-phase CO2",
    y=0.985, fontsize=15,
)
fig.text(
    0.5, 0.925,
    "Exploratory visualization only: observed gas signal is documented as unreliable",
    ha="center", va="center", color="crimson", fontsize=10,
)
fig.legend(
    handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.885),
    ncol=3, frameon=False, fontsize=9,
)
fig.tight_layout(rect=(0.02, 0.03, 0.985, 0.82), w_pad=2.0)
fig.savefig(
    analysis.PLOT_DIR / "external_gas_curve_comparison_LAB013_015.png",
    dpi=180, bbox_inches="tight",
)
plt.show()
display(external_gas_plot_audit.round(6))


### Exploratory dissolved-CO2 and DO comparison

Carbodoseur CO2 and model pool share mass-concentration units after conversion, and measured DO
and internal O2 share mg/L units. Nevertheless, neither pair has a validated observation function:
the model pool is a latent well-mixed balance with zero initial pool, and the internal O2 model
omits probe dynamics and reaeration. The overlays below are therefore exploratory and deliberately
carry no RMSE, R² or fitting weight. Brix and density remain contextual observations because no
validated mapping to an individual model state exists.


In [ ]:
exploratory_mapping = pd.DataFrame([
    {"observed": "Carbodoseur dissolved CO2 [g/L]", "model": "latent dissolved CO2 pool [g/L]",
     "direct_observation_function_validated": False, "scored": False},
    {"observed": "offline DO [mg/L]", "model": "internal depletion-only O2 [mg/L]",
     "direct_observation_function_validated": False, "scored": False},
    {"observed": "Brix / density", "model": "none", "direct_observation_function_validated": False, "scored": False},
    {"observed": "gas-phase CO2", "model": "qgas release", "direct_observation_function_validated": False,
     "scored": False, "note": "documented unreliable; loaded only for exploratory overlay"},
])
display(exploratory_mapping)

fig, axes = plt.subplots(3, 2, figsize=(13, 11), sharex=False)
for row, lab in enumerate(EXTERNAL_KINETIC_LABS):
    off = offline_13_15[offline_13_15["lab_id"].eq(lab)].sort_values("time_proxy_h")
    chain = external_chain_trajectories[external_chain_trajectories["batch"].eq(lab)]
    axes[row, 0].scatter(off["time_proxy_h"], pd.to_numeric(off["CO2_dissolved_actualT_mg_L"], errors="coerce") * 1e-3,
                         color="black", s=30, label="Carbodoseur measured")
    axes[row, 0].plot(chain["time_proxy_h"], chain["dissolved_CO2_pool_g_l"], color="tab:purple", lw=2.0, label="latent pool")
    axes[row, 1].scatter(off["time_proxy_h"], pd.to_numeric(off["DO_value"], errors="coerce"),
                         color="black", s=30, label="offline DO")
    axes[row, 1].plot(chain["time_proxy_h"], chain["o2_internal_mg_l"], color="tab:green", lw=2.0, label="internal O2")
    axes[row, 0].set_ylabel(f"{lab}\nCO2 [g/L]")
    axes[row, 1].set_ylabel(f"{lab}\nO2 [mg/L]")
    for ax in axes[row]: ax.grid(alpha=0.22)
axes[0, 0].legend(frameon=False); axes[0, 1].legend(frameon=False)
axes[-1, 0].set_xlabel("time from numbered sample 1 [proxy h]")
axes[-1, 1].set_xlabel("time from numbered sample 1 [proxy h]")
fig.suptitle("Exploratory only — no validated observation function and no fitting", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(analysis.PLOT_DIR / "external_exploratory_dissolved_CO2_DO_LAB013_015.png", dpi=180, bbox_inches="tight")
plt.show()

external_temperature_summary = external_temperature_inputs.groupby("batch", as_index=False).agg(
    temperature_min_c=("temperature_c", "min"),
    temperature_mean_c=("temperature_c", "mean"),
    temperature_max_c=("temperature_c", "max"),
)
conditions_and_activation = external_initial_conditions.merge(
    external_temperature_summary, on="batch", validate="one_to_one",
).merge(external_activation_summary, on="batch", validate="one_to_one")
descriptive_correlations = pd.DataFrame([
    {
        "candidate": candidate,
        "correlation_with_gate_start_proxy": conditions_and_activation[[candidate, "gate_start_proxy_h"]].corr().iloc[0, 1],
        "n": len(conditions_and_activation),
        "interpretation": "descriptive only; n=3 and inoculation t0 unresolved",
    }
    for candidate in ("temperature_mean_c", "X", "N", "G", "F")
])
display(conditions_and_activation.round(5))
display(descriptive_correlations.round(5))


### Artifacts, safeguards and interpretation

All LAB013–LAB015 outputs are diagnostic tables/figures. They are never added to the CO2
calibration objective. The assertions below preserve the existing fit membership and write an
explicit manifest. Old LAB016–LAB018 result files from an earlier execution, if present in the
results directory, are superseded and are not referenced by this notebook.


In [ ]:
assert fit_batches_b == ["LAB004", "LAB005", "LAB006", "LAB007", "LAB008", "LAB011"]
assert analysis.HOLDOUTS["natural"] == "LAB012"
assert SENSITIVITY_EXCLUDED_BATCH == "LAB010"
assert len(theta_natural_frozen) == 17
assert not any(lab in result["observations"]["batch"].unique() for lab in EXTERNAL_KINETIC_LABS)

tables_to_save = {
    "sensitivity_calibration_inventory.csv": sensitivity_inventory,
    "sensitivity_objectives_A_vs_B.csv": sensitivity_objectives,
    "sensitivity_parameters_A_vs_B.csv": sensitivity_parameters_long,
    "sensitivity_parameter_changes_A_vs_B.csv": sensitivity_parameter_changes,
    "sensitivity_batch_metrics_A_vs_B.csv": sensitivity_metrics_long,
    "sensitivity_batch_metric_changes_A_vs_B.csv": sensitivity_metric_changes,
    "sensitivity_identifiability_A_vs_B.csv": sensitivity_identifiability,
    "sensitivity_local_parameter_correlations_A_vs_B.csv": sensitivity_correlations,
    "sensitivity_activation_profiles_A_vs_B.csv": sensitivity_profiles,
    "sensitivity_loo_summary_A_vs_B.csv": sensitivity_loo_summary,
    "sensitivity_loo_estimates_A_vs_B.csv": sensitivity_loo_estimates,
    "signal_swap_observation_audit.csv": signal_swap_audit,
    "signals_repaired_vs_previous_parameters.csv": signal_parameters_long,
    "signals_repaired_vs_previous_parameter_changes.csv": signal_parameter_changes,
    "signals_repaired_vs_previous_objectives.csv": signal_objectives,
    "signals_repaired_vs_previous_batch_metrics.csv": signal_metrics_long,
    "signals_repaired_vs_previous_batch_metric_changes.csv": signal_metric_changes,
    "signals_repaired_vs_previous_identifiability.csv": signal_identifiability,
    "signals_repaired_vs_previous_local_correlations.csv": signal_correlations,
    "signals_repaired_vs_previous_loo_summary.csv": signal_loo_summary,
    "signals_repaired_vs_previous_loo_estimates.csv": signal_loo_estimates,
    "external_kinetic_holdout_source_audit_LAB013_015.csv": external_source_audit.drop(columns="path"),
    "external_kinetic_holdout_t0_audit_LAB013_015.csv": t0_audit,
    "external_kinetic_holdout_initial_conditions_LAB013_015.csv": external_initial_conditions,
    "external_kinetic_holdout_state_coverage_LAB013_015.csv": external_state_coverage,
    "external_kinetic_holdout_metrics_LAB013_015.csv": external_kinetic_metrics,
    "external_kinetic_holdout_point_errors_LAB013_015.csv": external_state_point_errors,
    "external_activation_brackets_LAB013_015.csv": external_activation_summary,
    "external_activation_chain_LAB013_015.csv": external_chain_trajectories,
    "external_kinetic_termination_LAB013_015.csv": external_termination_summary,
    "external_initial_conditions_activation_LAB013_015.csv": conditions_and_activation,
    "external_activation_descriptive_correlations_LAB013_015.csv": descriptive_correlations,
    "external_exploratory_mapping_LAB013_015.csv": exploratory_mapping,
    "external_exploratory_gas_CO2_hourly_LAB013_015.csv": external_gas_observed,
    "external_exploratory_gas_CO2_audit_LAB013_015.csv": external_gas_plot_audit,
}
SENSITIVITY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
analysis.PLOT_DIR.mkdir(parents=True, exist_ok=True)
for filename, frame in tables_to_save.items():
    frame.to_csv(SENSITIVITY_RESULTS_DIR / filename, index=False)


In [ ]:
from IPython.display import Markdown

mean_state = external_kinetic_metrics.groupby("state").agg(
    mean_nrmse=("nrmse_observed_range", "mean"), mean_bias=("bias_pred_minus_obs", "mean")
)
bracket_text = "; ".join(
    f"{row.batch}: [{row.chemical_activity_lower_proxy_h:.1f}, {row.chemical_activity_upper_proxy_h:.1f}] h (width {row.bracket_width_h:.1f} h)"
    for row in external_activation_summary.itertuples(index=False)
)
tail_text = "; ".join(
    f"{row.batch}: final G+F obs/pred {row.observed_final_GplusF_g_l:.1f}/{row.predicted_final_GplusF_g_l:.1f} g/L, last-12h qCO2/peak {row.mean_last12h_qCO2_over_peak:.2f}"
    for row in external_termination_summary.itertuples(index=False)
)
display(Markdown(f"""
## Frozen external diagnostic — interpretation boundaries

- **t0:** unresolved. All times are relative to numbered sample 1 and cannot establish absolute
  biological lag or onset from inoculation.
- **Chemical brackets:** {bracket_text}.
- **Kinetic reproduction:** use the state table above; mean range-normalized errors are
  `X={mean_state.loc['X', 'mean_nrmse']:.2f}`, `Xd={mean_state.loc['Xd', 'mean_nrmse']:.2f}`,
  `N={mean_state.loc['N', 'mean_nrmse']:.2f}`, `G={mean_state.loc['G', 'mean_nrmse']:.2f}`,
  `F={mean_state.loc['F', 'mean_nrmse']:.2f}`, `Gly={mean_state.loc['Gly', 'mean_nrmse']:.2f}`.
- **Termination:** {tail_text}.
- **What is supported:** frozen state/shape performance, bracket widths/order, and the modeled
  chain from state trajectories to latent release.
- **What is not supported:** gas CO2 RMSE/integral/amplitude, an absolute lag from inoculation,
  causal correlations from only three replicates, or parameter improvement claims.
- **LAB016–LAB018 implication:** these three measured brackets demonstrate that replacing missing
  chemistry by `[0, 0]` can by itself force premature activation. They do not prove that the
  kinetic model is otherwise correct; state residuals quantify that remaining question.
"""))


## Existing `no_lab010` sensitivity conclusion (unchanged calibration)

The repaired LAB004/LAB007/LAB008 signals retain their previously established effect on the
`no_lab010` fit and identifiability. LAB011 remains structurally underpredicted and LAB012 remains
the unchanged internal gas-phase CO2 holdout. This external diagnostic neither recalibrates nor
reinterprets those results; it only replaces the former LAB016–LAB018 gas comparison with a
frozen LAB013–LAB015 kinetic/activation assessment.
